# Singularity V2＋INT8映像VAE（int8_convrot） / Colab・Drive不要版

添付FastH3 V2 Notebookから分岐。元の起動方式・Pinggy接続・任意Drive保存・サンプル画像・4種類の操作画面を引き継ぎます。

- 20ステップ（ノイズ除去を20回計算）、Euler、動画shift12／音声shift3。通常Attentionを使用し、VSA（計算の間引き）は無効。
- INT8映像VAE（生成結果を映像へ戻す部品）を使用。Comfy-Org/MiniMax-H3 が2026-09-15に公開した `minimax_h3_video_vae_int8_convrot.safetensors` で、ファイル名やワークフロー名の「NewVAE」はこれを指します。Turbo LoRA（少ない計算回数向け追加モデル）は含みません。
- この20回設定は今回のローカル比較で用いた開始設定です。作者唯一の推奨設定という意味ではありません。
- 必ず別の新GPUランタイムで開始。約24GB以上のGPUメモリを想定、22GiB未満やT4は停止。メモリの多いA100級からの試験を勧めますが、割当や成功は保証しません。
- まずR2Vアプリ版の0.4MP・5秒で確認。モデルだけで約38GiBを取得するため、作業ディスクにも余裕が必要です。
- Notebook構文と全4グラフの設定・接続はローカル検査済み。Colabでの導入・トンネル・生成は未試験です。

制作：ざすこ（道草 雑草子）／[AIみちくさch](https://www.youtube.com/channel/UC84fyKjiilxssZVxhE_RiaA)

## このNotebookの保存方式
モデルは毎回Colab一時ディスクへダウンロード。Drive不要。出力のDrive保存は任意です。

## 🔗 事前準備：Pinggy Token の設定
このNotebookは、Google Colab上のComfyUIを外部（ブラウザ・PC側ツール等）から使えるように、**Pinggy**というトンネルサービスで一時的な公開URLを発行します。有料（Pro）トークンを登録すると時間制限なしで使えます。トークンが無い場合は、無料のPinggy（1回60分まで）で接続します。

> 💡 **Pinggyとは**：Colabのサーバーは通常インターネットから直接アクセスできないため、Pinggyが「Colab内で動いているComfyUI」と「あなたのブラウザ」をつなぐ一時的な公開URL（トンネル）を作ってくれます。このURLは**このColabセッション限りの一時URL**で、Notebookを再起動すると変わります。

**① Pinggyアカウント・トークンの取得**
1. [https://pinggy.io](https://pinggy.io) でアカウントを作成（この接続コードはPro接続先を使用。契約・制限はPinggy側で確認してください）
2. ダッシュボードから接続トークン（アクセストークン）を取得してコピー

**② Google Colab シークレットへの登録**
1. 左サイドバーの 🔑 **シークレット**（鍵アイコン）を開く
2. **＋ 新しいシークレットを追加** をクリック
3. 名前に `PINGGY_TOKEN`、値にコピーしたトークンを貼り付けて保存
4. **ノートブックからのアクセス** をオンにする

> ⚠️ トークンはシークレットで管理し、コードに直接貼り付けないでください。


# 起動
GPUランタイムに変更し、次のセルを実行。Pinggyの有料トークンがあれば、シークレット `PINGGY_TOKEN` に登録しておきます（無ければ無料のPinggyで、1回60分まで接続します）。
終了後の画面でUser workflows > Singularity_V2_NewVAE > r2v_appを選びます。


In [ ]:
import base64
# @markdown # 👈この（▶）ボタンをクリックしてComfyUIを起動（初回はモデル取得と導入に時間がかかります）
# @markdown ### ※停止と再起動もこのボタンで行います。
# @markdown ## 🌱Pinggy接続後、実行ログに表示される青い「ComfyUIを開く」ボタンをクリックしてください。

# ==========================================
# 🔧 PyTorch系ライブラリの軽量チェック（再起動ループ回避版）
# ==========================================
# 重要：このセル内では、現在のColabカーネルに torch / triton を直接 import しません。
# 理由：torchをimportすると内部でtritonも読み込まれ、その後pipがtriton周辺を確認しただけで
#       Colabが「セッションを再起動してください」と表示しやすくなるためです。
# 方針：別プロセスで確認し、必要な場合だけ修復します。
import sys, subprocess

CORE_TORCH_PACKAGES = [
    "torch==2.9.0",
    "torchvision==0.24.0",
    "torchaudio==2.9.0",
]


def run(cmd):
    print("$", " ".join(cmd))
    subprocess.check_call(cmd)


def torch_stack_ok():
    check_code = r'''
import sys
try:
    import torch, torchvision, torchaudio
    print("torch:", torch.__version__)
    print("torchvision:", torchvision.__version__)
    print("torchaudio:", torchaudio.__version__)
    print("torch cuda:", torch.version.cuda)
    print("cuda available:", torch.cuda.is_available())
    ok = (torch.__version__ == "2.9.0+cu130" and torchvision.__version__ == "0.24.0+cu130" and torchaudio.__version__ == "2.9.0+cu130" and torch.cuda.is_available())
    sys.exit(0 if ok else 2)
except Exception as e:
    print("PyTorch check failed:", repr(e))
    sys.exit(1)
'''
    result = subprocess.run([sys.executable, "-c", check_code], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    return result.returncode == 0

if not torch_stack_ok():
    print("=" * 70)
    print("🔧 torch / torchvision / torchaudio を CUDA 13.0版(cu130)にそろえます")
    print("※必要な時だけ修復します。修復後に「セッションを再起動してください」と表示された場合は、")
    print("　その時だけ再起動し、このセルより上には戻らずこのセルから再実行してください。")
    print("=" * 70)
    run([sys.executable, "-m", "pip", "install", "--no-cache-dir", *CORE_TORCH_PACKAGES,
         "--index-url", "https://download.pytorch.org/whl/cu130"])
    if not torch_stack_ok():
        raise RuntimeError("torch / torchvision / torchaudio の修復に失敗しました。ランタイムを新規にして再実行してください。")

print("✅ PyTorch系ライブラリ確認OK。このままComfyUI起動処理に進みます。")

# GPU名の限定ではなく、実際の計算機能とメモリを確認します。
# @markdown ### GPU自動確認（L4 / A100 / H100 / G4系など）
# @markdown G4はRTX PRO 6000 Blackwell系。起動時に実際に割り当てられたGPUを表示します。
import json as _gpu_json
def _validate_gpu_info(info):
    if tuple(info["capability"]) < (8, 0):
        raise RuntimeError("このBF16・量子化モデル構成はCompute Capability 8.0以上を対象とします。T4は対象外です。GPUを変更してください。")
    if info["total_gib"] < 22:
        raise RuntimeError("このNotebookでは約24GB以上のGPUメモリを想定しています。小容量の分割GPUの場合も別GPUへ変更してください。")
    return info["total_gib"] <= 26
_gpu_probe = r"""
import json, torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPUが利用できません。ColabでGPUランタイムを選択してください。")
device = torch.cuda.current_device()
p = torch.cuda.get_device_properties(device)
a = torch.ones((64, 64), device=device, dtype=torch.bfloat16)
b = a @ a
torch.cuda.synchronize()
assert torch.isfinite(b).all().item(), "GPUの小規模計算に失敗しました"
print(json.dumps({"name":p.name,"capability":list(torch.cuda.get_device_capability(device)),"total_gib":p.total_memory/1024**3,"torch":torch.__version__,"cuda":torch.version.cuda}))
"""
_gp = subprocess.run([sys.executable, "-c", _gpu_probe],capture_output=True,text=True)
if _gp.returncode:
    raise RuntimeError("GPU/CUDAの初期確認に失敗しました。新しいGPUランタイムで再実行してください。\n" + _gp.stdout + _gp.stderr)
_gpu_info = _gpu_json.loads(_gp.stdout.strip().splitlines()[-1])
using_L4_GPU = _validate_gpu_info(_gpu_info)  # 既存の24GB向け起動分岐をメモリ量で選択
print("割り当てGPU:", _gpu_info)
print("GPU初期確認OK。続いて通常Attentionの小規模計算を確認します。動画生成の成功・速度保証ではありません。")

include_manager = True # @param {type:"boolean"}

# ==========================================
# 📁 Google Drive マウント
# ==========================================
# @markdown ## 📁 Google Driveに接続
# @markdown ComfyUIの出力ファイルをバックアップする場合はチェックを入れてください

# @markdown ---
# @markdown ### Google Driveを使用する
use_google_drive = True  # @param {type:"boolean"}

# @markdown ---

if use_google_drive:
    from google.colab import drive
    _drive_mount_ok = False
    for _mount_attempt in range(3):
        try:
            drive.mount('/content/drive', force_remount=(_mount_attempt > 0))
            _drive_mount_ok = True
            break
        except Exception as _mount_err:
            print(f"⚠️ Google Driveのマウントに失敗しました（{_mount_attempt + 1}/3回目）: {_mount_err}")
            print("   Googleの認証ポップアップが出ていたら、許可まで進めてから少し待ってください。")
            import time as _mount_time
            _mount_time.sleep(5)
    if not _drive_mount_ok:
        raise RuntimeError(
            "Google Driveのマウントに3回失敗しました。\n"
            "『ランタイム』→『ランタイムを接続解除して削除』後にもう一度実行するか、\n"
            "ブラウザのポップアップブロック・サードパーティCookie設定を確認してください。"
        )

    print("=" * 70)
    print("✅ Google Driveがマウントされました")
    print("📁 マウント先: /content/drive/MyDrive")
    print("=" * 70)
else:
    print("=" * 70)
    print("⏸️  Google Driveは使用しません")
    print("💡 出力ファイルは/content/ComfyUI/outputに保存されます")
    print("   （ランタイム終了時に消えるので注意してください）")
    print("=" * 70)


# ==========================================
# 📤 Google Drive 出力設定
# ==========================================
# @markdown ## 📤 Google Drive 出力設定
# @markdown ### ComfyUIの出力先をGoogle Driveに変更する
enable_gdrive_output = True  # @param {type:"boolean"}

# @markdown ### 出力先フォルダ設定
GDRIVE_OUTPUT = "/content/drive/MyDrive/ComfyUI_output"  # @param {type:"string"}

# @markdown ---

# ==========================================
# 🔗 Pinggy 接続設定
# ==========================================
# @markdown ## 🔗 Pinggy 接続設定
# @markdown ### Pinggyの有料トークン（Pro）を使う
# @markdown オン：シークレット `PINGGY_TOKEN` のトークンで Pinggy Pro につなぐ（時間制限なし）。トークンが読めないときは無料でつなぐ。<br>
# @markdown オフ：トークンなしで無料の Pinggy につなぐ（1回60分まで。切れたらこのセルを再実行）。
use_pinggy_pro = True  # @param {type:"boolean"}

# @markdown ---

# ★ トークンはここ（メインのスレッド）で読む。
# 裏のスレッドで読むと、Colabがシークレットの許可画面を出せずに
# 「Secrets can only be fetched when running from the Colab UI」で失敗する。
# 初回は「ノートブックにアクセスを許可しますか」と聞かれるので、許可する。
PINGGY_TOKEN = ""
if use_pinggy_pro:
    try:
        from google.colab import userdata
        PINGGY_TOKEN = (userdata.get("PINGGY_TOKEN") or "").strip()
    except Exception as _pinggy_err:
        print(f"⚠️ PINGGY_TOKEN を読めませんでした（{_pinggy_err}）")
    if PINGGY_TOKEN:
        print("✅ PINGGY_TOKEN を読み込みました。Pinggy Pro でつなぎます。")
    else:
        print("ℹ️ 有料トークンが読めなかったので、無料の Pinggy（1回60分まで）でつなぎます。")
        print("   Proで使うなら、左の🔑シークレットに PINGGY_TOKEN を登録し「ノートブックからのアクセス」をオンにして再実行してください。")
else:
    print("ℹ️ 無料の Pinggy（1回60分まで）でつなぎます。")

# 出力フォルダの作成
if use_google_drive and enable_gdrive_output:
    from pathlib import Path
    output_path = Path(GDRIVE_OUTPUT)

    if output_path.exists():
        print(f"📁 既存のフォルダを使用: {GDRIVE_OUTPUT}")
    else:
        output_path.mkdir(parents=True, exist_ok=True)
        print(f"📁 新規フォルダを作成: {GDRIVE_OUTPUT}")

    print("=" * 70)
    print("✅ ComfyUIの出力先がGoogle Driveに設定されます")
    print(f"📁 保存先: {GDRIVE_OUTPUT}")
    print("💡 生成したファイルが直接Google Driveに保存されます")
    print("=" * 70)
elif enable_gdrive_output and not use_google_drive:
    print("=" * 70)
    print("⚠️ Google Drive出力が有効ですが、Google Driveがマウントされていません")
    print("💡 use_google_driveにチェックを入れてください")
    print("=" * 70)
else:
    print("=" * 70)
    print("⏸️  Google Drive出力は無効です")
    print("💡 出力ファイルは/content/ComfyUI/outputに保存されます")
    print("=" * 70)


%cd /content
from IPython.display import clear_output
clear_output()
# torch / triton はColab標準または上のチェック結果を使います。
# xformersだけを --no-deps で入れ、torch/tritonを勝手に入れ替えないようにします。
!python -m pip install -q torchsde einops diffusers accelerate
# cu130 experiment: xformers 0.0.32.post1 is built for cu128, compatibility with cu130 unconfirmed.
# Skipping to avoid conflicts (ComfyUI runs fine without xformers).
# !python -m pip install -q --no-deps xformers==0.0.32.post1
# pipがtorch/triton系を別バージョンへ動かさないための制約ファイル
# Colabの通常Pythonセルでは bash の here-document が壊れやすいため、Pythonで安全に書き出します。
def _installed_version(_name):
    try:
        import importlib.metadata as _md
        return _md.version(_name)
    except Exception:
        return None
_np_ver = _installed_version('numpy')
_pb_ver = _installed_version('protobuf')
_constraints = ['torch==2.9.0', 'torchvision==0.24.0', 'torchaudio==2.9.0', 'triton==3.5.0']
if _np_ver: _constraints.append('numpy==' + _np_ver)
if _pb_ver: _constraints.append('protobuf==' + _pb_ver)
with open('/content/torch_core_constraints.txt', 'w', encoding='utf-8') as f:
    f.write(chr(10).join(_constraints) + chr(10))
print('🔒 バージョン固定:', ' / '.join(_constraints))
!pip install av spandrel albumentations onnx opencv-python onnxruntime -c /content/torch_core_constraints.txt
!pip install color-matcher -c /content/torch_core_constraints.txt
print("ℹ️ protobufはColab既定のまま使います")
!pip install onnxruntime-gpu -c /content/torch_core_constraints.txt
# Singularity: 元Notebookと同じComfyUI v0.36.0固定版。Colab実機は未検証。
FASTH3_COMFYUI_COMMIT = "ee71d5c4993f29086b27fde1629a945ae48425bf"
from pathlib import Path
import os as _os_cl, shutil as _sh_cl, tarfile as _tarfile, urllib.request as _request
_COMFY = "/content/ComfyUI"
_marker = _os_cl.path.join(_COMFY, "FASTH3_COMMIT.txt")
if _os_cl.path.exists(_os_cl.path.join(_COMFY, "main.py")):
    _head = open(_marker).read().strip() if _os_cl.path.exists(_marker) else ""
    if _head != FASTH3_COMFYUI_COMMIT:
        raise RuntimeError("別バージョンのComfyUIが残っています。『ランタイムを接続解除して削除』後、新しいランタイムで実行してください。Driveのモデルは削除しません。")
else:
    _TMP = "/content/_fasth3_v2_source"
    _TAR = "/content/comfyui_v2.tar.gz"
    _sh_cl.rmtree(_TMP, ignore_errors=True)
    _os_cl.makedirs(_TMP, exist_ok=True)
    _request.urlretrieve(f"https://codeload.github.com/Comfy-Org/ComfyUI/tar.gz/{FASTH3_COMFYUI_COMMIT}", _TAR)
    with _tarfile.open(_TAR) as _tf:
        _tf.extractall(_TMP, filter="data")
    _roots = [p for p in Path(_TMP).iterdir() if (p / "main.py").is_file()]
    if len(_roots) != 1:
        raise RuntimeError("ComfyUIの取得に失敗しました。")
    _os_cl.makedirs(_COMFY, exist_ok=True)
    for _item in _roots[0].iterdir():
        _dst = Path(_COMFY) / _item.name
        if _dst.exists():
            raise RuntimeError(f"既存ファイルと競合しています: {_dst}。新しいランタイムで実行してください。")
        _sh_cl.move(str(_item), str(_dst))
    Path(_marker).write_text(FASTH3_COMFYUI_COMMIT)
    _sh_cl.rmtree(_TMP)
    _os_cl.remove(_TAR)
for _required in ["comfy_extras/nodes_minimax_h3.py", "comfy_extras/nodes_sparse_attention.py"]:
    if not (Path(_COMFY) / _required).is_file():
        raise RuntimeError(f"必要な標準ノードがありません: {_required}")
print("ComfyUI v0.36.0:", FASTH3_COMFYUI_COMMIT)
%cd /content
run([sys.executable, "-m", "pip", "install", "-r", "/content/ComfyUI/requirements.txt", "-c", "/content/torch_core_constraints.txt"])
clear_output()

!mkdir -p /content/ComfyUI/custom_nodes
%cd /content/ComfyUI/custom_nodes
# 🌱 MiniMax H3の公式ワークフローはComfyUIコアだけで完結し、カスタムノードは不要と確認済み
# （ComfyMathExpression含む全ノードがcomfy_extras=コア機能）。起動を速くするため、
# 以前入れていた汎用カスタムノード群（GGUF/VideoHelperSuite/KJNodes/rgthree/essentials/
# LogicUtils/VFI/Frame-Interpolation/PainterI2V）は撤去し、ComfyUI-Managerだけ任意で残す。
if include_manager:
    !git clone https://github.com/ltdrdata/ComfyUI-Manager

# 通常Attentionを使用。comfy-kitchenは量子化モデル等の共通部品として残します。
run([sys.executable, "-m", "pip", "install", "comfy-kitchen==0.2.34", "comfy-aimdo==0.5.3", "-c", "/content/torch_core_constraints.txt"])
_GPU_KERNEL_PROBE = 'import torch\nq = torch.randn((1, 2, 256, 128), device="cuda", dtype=torch.bfloat16)\nresult = torch.nn.functional.scaled_dot_product_attention(q, q, q)\ntorch.cuda.synchronize()\nassert result.shape == q.shape and torch.isfinite(result).all().item(), "通常Attentionの小規模計算に失敗"\nprint("GPU / PyTorch dense attention: PASS")\n'
_kp = subprocess.run([sys.executable, "-c", _GPU_KERNEL_PROBE],capture_output=True,text=True)
print(_kp.stdout)
if _kp.returncode:
    raise RuntimeError("通常AttentionのGPU実行に失敗しました。大量のモデルをダウンロードする前に停止します。\n" + _kp.stderr)
if include_manager:
    %cd /content/ComfyUI/custom_nodes/ComfyUI-Manager
    !pip install -r requirements.txt -c /content/torch_core_constraints.txt

# 🆕 protobufはColab既定のものをそのまま使う。
# 以前ここで protobuf==5.29.1 を入れ直していたが、このセルが先に書いた制約ファイル
# （Colab既定の 5.29.6 で固定）と衝突して ResolutionImpossible で失敗していた。

clear_output()

%cd /content/ComfyUI

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import subprocess
import sys
from pathlib import Path

def install_apt_packages():
    packages = ['aria2']

    try:
        # Run apt install silently (using -qq)
        subprocess.run(
            ['apt-get', '-y', 'install', '-qq'] + packages,
            check=True,
            capture_output=True
        )
        print("✓ apt packages installed")
    except subprocess.CalledProcessError as e:
        print(f"✗ Error installing apt packages: {e.stderr.decode().strip() or 'Unknown error'}")

print("Installing apt packages...")
install_apt_packages()

def download_with_aria2c(link, folder="/content/ComfyUI/models/loras", hf_token=None):
    import os
    from urllib.parse import urlparse, unquote

    # Hugging Faceなど、URLの末尾に正式なファイル名が入っている通常リンク用
    path_name = os.path.basename(urlparse(link).path)
    filename = unquote(path_name) if path_name else "downloaded_model.safetensors"
    header_arg = ""
    if hf_token and "huggingface.co" in link.lower():
        header_arg = f' --header="Authorization: Bearer {hf_token}"'
    command = f"aria2c --console-log-level=error -c -x 16 -s 16 -k 1M{header_arg} \"{link}\" -d \"{folder}\" -o \"{filename}\""

    print("Executing download command:")
    print(command.replace(hf_token, "***") if hf_token else command)

    os.makedirs(folder, exist_ok=True)
    get_ipython().system(command)

    return filename

def add_civitai_token_to_url(civitai_link, civitai_token=None):
    """CivitaiのURLにtokenを安全に追加する。既にtokenがある場合はそのまま使う。"""
    from urllib.parse import urlparse, parse_qsl, urlencode, urlunparse

    parsed = urlparse(civitai_link)
    if not parsed.scheme or not parsed.netloc:
        raise ValueError("Invalid Civitai URL format. Please use a link like: https://civitai.com/api/download/models/1523247?...")

    query = dict(parse_qsl(parsed.query, keep_blank_values=True))
    if civitai_token and "token" not in query:
        query["token"] = civitai_token

    return urlunparse((parsed.scheme, parsed.netloc, parsed.path, parsed.params, urlencode(query), parsed.fragment))

def pick_new_downloaded_file(folder, before_files):
    """wget実行後に増えたファイルから、実体のある最新ファイルを推定する。"""
    import os

    after_files = set(os.listdir(folder))
    candidates = []
    for name in after_files - before_files:
        path = os.path.join(folder, name)
        if os.path.isfile(path) and os.path.getsize(path) > 0:
            # wgetの一時ファイルやログっぽいものは除外
            if not name.endswith((".tmp", ".aria2")):
                candidates.append(path)

    if not candidates:
        return None

    candidates.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    return candidates[0]

def download_civitai_model(civitai_link, civitai_token, folder="/content/ComfyUI/models/loras"):
    """
    Civitai専用ダウンロード。
    重要：-Oで固定名を付けず、wgetの --content-disposition / --trust-server-names を使う。
    これにより、Civitaiが返す正式ファイル名（例：FlowCamera_epoch67.safetensors）で保存する。
    """
    import os
    import subprocess

    os.makedirs(folder, exist_ok=True)
    civitai_url = add_civitai_token_to_url(civitai_link, civitai_token)

    before_files = set(os.listdir(folder))

    cmd = [
        "wget",
        "--max-redirect=10",
        "--content-disposition",
        "--trust-server-names",
        "--show-progress",
        "-P", folder,
        civitai_url,
    ]

    print("Downloading from Civitai with official filename...")
    print("$ " + " ".join([f'\"{c}\"' if " " in c or "&" in c or "?" in c else c for c in cmd]))

    result = subprocess.run(cmd, text=True)
    if result.returncode != 0:
        print("❌ Civitai download failed.")
        return False

    downloaded_path = pick_new_downloaded_file(folder, before_files)

    # 同名ファイルが既に存在していて新規差分が取れない場合の保険：フォルダ内の最新safetensorsを表示
    if downloaded_path is None:
        safetensors = [
            os.path.join(folder, f) for f in os.listdir(folder)
            if f.lower().endswith((".safetensors", ".ckpt", ".pt", ".pth", ".bin")) and os.path.isfile(os.path.join(folder, f))
        ]
        if safetensors:
            downloaded_path = max(safetensors, key=os.path.getmtime)

    if downloaded_path and os.path.exists(downloaded_path) and os.path.getsize(downloaded_path) > 0:
        filename = os.path.basename(downloaded_path)
        print(f"✅ Civitai model downloaded successfully: {downloaded_path}")
        print(f"📛 Saved filename: {filename}")
        return filename

    print("❌ Civitai download finished, but the saved file could not be detected.")
    return False

def download_lora(link, folder="/content/ComfyUI/models/loras", civitai_token=None):
    """
    Download a model file, automatically detecting if it's a Civitai link or huggingface download.

    Args:
        link: The download URL (either huggingface or Civitai)
        folder: Destination folder for the download
        civitai_token: Optional token for Civitai downloads (required if link is from Civitai)

    Returns:
        The filename of the downloaded model
    """
    if "civitai.com" in link.lower():
        if not civitai_token:
            print("⚠️ Civitaiトークンが未設定のため、トークンなしでダウンロードを試します。")
            print("   非公開・年齢制限・ログイン必須モデルの場合は失敗することがあります。")
        return download_civitai_model(link, civitai_token, folder)
    else:
        return download_with_aria2c(link, folder)

def model_download(url: str, dest_dir: str, filename: str = None, silent: bool = True, hf_token: str = None) -> bool:
    """
    Colab-optimized download with aria2c

    Args:
        url: Download URL
        dest_dir: Target directory (will be created if needed)
        filename: Optional output filename (defaults to URL filename)
        silent: If True, suppresses all output (except errors)
        hf_token: Optional Hugging Face token. Added as an Authorization header
            only when the URL host is huggingface.co (rate-limit avoidance).

    Returns:
        bool: True if successful, False if failed
    """
    try:
        # Create destination directory
        Path(dest_dir).mkdir(parents=True, exist_ok=True)

        # Set filename if not specified
        if filename is None:
            filename = url.split('/')[-1].split('?')[0]  # Remove URL parameters

        # Build command
        cmd = [
            'aria2c',
            '--console-log-level=error',
            '-c', '-x', '16', '-s', '16', '-k', '1M',
            '-d', dest_dir,
            '-o', filename,
            url
        ]
        if hf_token and "huggingface.co" in url.lower():
            cmd.insert(1, f'--header=Authorization: Bearer {hf_token}')

        # Add silent flags if requested
        if silent:
            cmd.extend(['--summary-interval=0', '--quiet'])
            print(f"Downloading {filename}...", end=' ', flush=True)

        # Run download
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)

        if silent:
            print("Done!")
        else:
            print(f"Downloaded {filename} to {dest_dir}")
        return filename

    except subprocess.CalledProcessError as e:
        error = e.stderr.strip() or "Unknown error"
        print(f"\nError downloading {filename}: {error}")
        return False
    except Exception as e:
        print(f"\nError: {str(e)}")
        return False

# ========================================
# ダウンロード処理関数
# ========================================

def parse_urls(url_string):
    """
    複数行のURL文字列をパースして、有効なURLのリストを返す

    Args:
        url_string: 改行区切りのURL文字列

    Returns:
        list: 有効なURLのリスト
    """
    urls = []
    for line in url_string.strip().split('\n'):
        line = line.strip()
        # コメント行（#始まり）と空行をスキップ
        if line and not line.startswith('#') and line.startswith('http'):
            urls.append(line)
    return urls

# ========================================
# 🌱💾 Google Driveへのモデル永続保存
# ========================================
# H3のモデル本体（合計約40〜60GB）を、毎回Colabへダウンロードし直すのではなく
# Google Driveの下記フォルダに保存し、2回目以降はそれを再利用する。
# 初回だけダウンロードが発生し、2回目以降はセットアップがぐっと速くなる。

# @markdown ---
# @markdown ## 💾 Google Driveへのモデル永続保存
# @markdown モデル本体をGoogle Driveの以下のフォルダに保存し、次回以降は再ダウンロードをスキップします。
DRIVE_MODEL_ROOT = "/content/drive/MyDrive/MiniMaxH3_Models"  # @param {type:"string"}
# @markdown 容量が気になる方はフォルダパスを変更してください（フォルダは自動作成されます）。
# @markdown チェックを外すと、Driveに保存せず毎回ローカルへダウンロードする従来動作になります（Colab終了で消えます）。
persist_models_to_drive = False  # @param {type:"boolean"}

# @markdown ---
# @markdown ## 💾 モデル読み込み方式（既定: ステージングなし・Drive保存時のみ適用）
# @markdown Driveのモデルファイルへ直接アクセスします（ローカルへのコピー待ち時間なし）。基本的にONのままで問題ありません。チェックを外すと、従来どおり一度ローカルディスクへコピーしてから使います。
skip_local_staging = True  # @param {type:"boolean"}

import os
import shutil
from pathlib import Path

# 既知のH3公式ファイルは、想定サイズ(GiB)を下回っていたら壊れているとみなして再ダウンロードする
MODEL_MIN_GIB_HINTS = {
    "minimax_h3_ref2va_pruned_int8_convrot.safetensors": 19.0,
    "minimax_h3_fl2va_pruned_int8_convrot.safetensors": 19.0,
    "minimax_h3_ref2va_int8_convrot.safetensors": 19.0,
    "minimax_h3_fl2va_int8_convrot.safetensors": 19.0,
    "Minimax-h3_Singularity_ref2va_v2_pruned_int8.safetensors": 19.0,
    "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors": 14.0,
    "qwen3vl_32b_minimax_h3_int8_convrot.safetensors": 30.0,
    "minimax_h3_video_vae_int8_convrot.safetensors": 2.6,
    "minimax_h3_audio_vae_fp32.safetensors": 0.5,
}


def ensure_model_persistent(url, comfy_folder, filename=None, min_expected_gib=None, civitai_token=None, hf_token=None):
    """
    Google Driveに保存したモデルを再利用しつつ、必要な時だけダウンロードする。
    さらに、実行時の読み込み速度のためColabのローカルディスクへステージング（コピー）してから使う。

    1. Drive上に既に十分なサイズのファイルがあれば、再ダウンロードしない。
       無ければ、まずColabのローカルディスクへダウンロードし、サイズを確認してからDriveへ移動する。
    2. ComfyUIのmodelsフォルダへは、Driveのファイルを直接シンボリックリンクするのではなく、
       いったんColabのローカルディスク（/content/h3_model_stage/）へコピーしてから、
       そのローカルコピーへシンボリックリンクを張る。
       Google Drive越しの読み込み（FUSEマウント）はネットワーク経由になり遅くなりがちなので、
       実際にComfyUIが読みに行く先はローカルSSD相当にすることで高速化する。
       ローカルの空き容量が足りない場合は、従来通りDriveへ直接シンボリックリンクする。
    """
    from urllib.parse import urlparse, unquote

    comfy_dest_dir = Path(comfy_folder)
    comfy_dest_dir.mkdir(parents=True, exist_ok=True)

    is_civitai = "civitai.com" in url.lower()

    if not filename and not is_civitai:
        path_name = os.path.basename(urlparse(url).path)
        filename = unquote(path_name) if path_name else "model.safetensors"

    if not persist_models_to_drive or is_civitai:
        # Drive保存なし（従来通りローカルへ直接ダウンロード）。
        # CivitaiはダウンロードするまでファイルGname確定しないため、Drive方式の対象外にしている。
        if is_civitai:
            download_lora(url, folder=str(comfy_dest_dir), civitai_token=civitai_token)
        else:
            model_download(url, str(comfy_dest_dir), filename=filename)
        return

    if min_expected_gib is None:
        min_expected_gib = MODEL_MIN_GIB_HINTS.get(filename)

    drive_dir = Path(DRIVE_MODEL_ROOT)
    drive_dir.mkdir(parents=True, exist_ok=True)
    drive_path = drive_dir / filename
    comfy_path = comfy_dest_dir / filename

    min_expected_bytes = int(min_expected_gib * (1024 ** 3) * 0.98) if min_expected_gib else 1

    if drive_path.is_file() and drive_path.stat().st_size >= min_expected_bytes:
        print(f"✅ Driveに保存済みのモデルを再利用: {filename}（{drive_path.stat().st_size / 1024**3:.2f} GiB）")
    else:
        print(f"⬇️ Driveに未保存のため新規ダウンロード: {filename}")
        tmp_dir = Path("/content/h3_model_tmp")
        tmp_dir.mkdir(parents=True, exist_ok=True)
        import time as _dl_time_mod
        _dl_started = _dl_time_mod.time()
        tmp_filename = model_download(url, str(tmp_dir), filename=filename, hf_token=hf_token)
        _dl_elapsed = _dl_time_mod.time() - _dl_started
        tmp_path = tmp_dir / (tmp_filename or filename)
        if not tmp_path.is_file():
            print(f"❌ ダウンロードに失敗しました: {filename}")
            return
        if min_expected_gib and tmp_path.stat().st_size < min_expected_bytes:
            print(f"⚠️ ファイルサイズが想定より小さいです（{tmp_path.stat().st_size / 1024**3:.2f} GiB）。破損の可能性があるため保存を中止します。")
            return
        _dl_mbps = (tmp_path.stat().st_size / 1024 / 1024) / _dl_elapsed if _dl_elapsed > 0 else 0
        print(f"⬇️ ダウンロード完了: {filename}（{_dl_elapsed:.1f}秒、{_dl_mbps:.1f} MB/s、aria2c 16並列）")
        print(f"📦 Google Driveへ移動中...（{tmp_path.stat().st_size / 1024**3:.2f} GiB、少し時間がかかります）")
        if drive_path.exists():
            drive_path.unlink()
        shutil.move(str(tmp_path), str(drive_path))
        print(f"✅ Driveへ保存しました: {drive_path}")

    # ── ここからローカルステージング ──
    stage_dir = Path("/content/h3_model_stage")
    stage_dir.mkdir(parents=True, exist_ok=True)
    stage_path = stage_dir / filename
    drive_size = drive_path.stat().st_size

    link_target = drive_path  # 既定はDriveへ直接（空き容量不足などの保険）

    stage_ok = stage_path.is_file() and not stage_path.is_symlink() and stage_path.stat().st_size == drive_size
    if skip_local_staging:
        print(f"🧪 実験: ローカルステージングをスキップし、Driveへ直接シンボリックリンクします: {filename}")
        link_target = drive_path
    elif stage_ok:
        print(f"✅ ローカルステージング済みを再利用: {filename}")
        link_target = stage_path
    else:
        free_gib = shutil.disk_usage("/content").free / (1024 ** 3)
        needed_gib = drive_size / (1024 ** 3) + 5  # 5GiBの余裕を見る
        if free_gib < needed_gib:
            print(f"⚠️ ローカルディスクの空きが足りないため（空き{free_gib:.1f}GiB／必要{needed_gib:.1f}GiB）、Driveから直接読み込みます: {filename}")
        else:
            print(f"📥 ローカルディスクへステージング中...（{drive_size / 1024**3:.2f} GiB、少し時間がかかります）")
            stage_tmp = stage_path.with_suffix(stage_path.suffix + ".copying")
            import time as _stage_time_mod
            _stage_started = _stage_time_mod.time()
            with open(drive_path, "rb") as _src, open(stage_tmp, "wb") as _dst:
                shutil.copyfileobj(_src, _dst, length=64 * 1024 * 1024)
            _stage_elapsed = _stage_time_mod.time() - _stage_started
            _stage_mbps = (drive_size / 1024 / 1024) / _stage_elapsed if _stage_elapsed > 0 else 0
            stage_tmp.replace(stage_path)
            print(f"✅ ローカルステージング完了: {filename}（{_stage_elapsed:.1f}秒、{_stage_mbps:.1f} MB/s）")
            link_target = stage_path

    if comfy_path.is_symlink():
        if comfy_path.resolve() != link_target.resolve():
            comfy_path.unlink()
            comfy_path.symlink_to(link_target)
    elif comfy_path.exists():
        print(f"⚠️ {comfy_path} は既にシンボリックリンク以外のファイルとして存在するため、そのままにします。")
    else:
        comfy_path.symlink_to(link_target)


def download_models_to_folder(url_string, folder_path, civitai_token=None):
    """
    指定されたフォルダに複数のモデルをダウンロード（Drive永続化対応版）。

    Args:
        url_string: 改行区切りのURL文字列
        folder_path: ダウンロード先フォルダパス
        civitai_token: Civitai APIトークン（オプション）
    """
    urls = parse_urls(url_string)

    if not urls:
        return

    print(f"\n📂 対象フォルダ: {folder_path}")
    print(f"   ファイル数: {len(urls)}")

    for i, url in enumerate(urls, 1):
        if "civitai.com" in url.lower():
            display_name = "Civitai公式ファイル名（ダウンロード後に確定）"
            hint_key = None
        else:
            display_name = url.split('/')[-1].split('?')[0]
            hint_key = display_name
        print(f"   [{i}/{len(urls)}] {display_name}")
        ensure_model_persistent(
            url,
            folder_path,
            filename=None if "civitai.com" in url.lower() else display_name,
            min_expected_gib=MODEL_MIN_GIB_HINTS.get(hint_key) if hint_key else None,
            civitai_token=civitai_token,
        )

    print(f"✓ 完了: {folder_path}\n")


# ========================================
# 📥 各種モデルのダウンロード設定
# ========================================

# @markdown # 📥 モデルダウンロード設定
# @markdown 各フォルダごとに、使用したいモデルのダウンロードURLを入力してください。
# @markdown 複数のURLは改行で区切って入力できます。

# @markdown ---
# @markdown ## 🔑 Civitai API Token（任意・H3本体では不要）
# @markdown H3のモデル本体はHugging Faceから取得するため、この設定はH3を使うだけなら不要です。Civitai経由で追加のカスタムモデルをDLしたい場合のみ、`CIVITAI_KEY`という名前でColabの🔑シークレットに登録してください。

use_civitai_key = False  # @param {type:"boolean"}
# @markdown ↑ オンのときだけ、シークレット `CIVITAI_KEY` を読みにいきます（オフならシークレットの確認を飛ばします）。

civitai_token = ""

try:
    from google.colab import userdata
    if use_civitai_key:
        civitai_token = userdata.get('CIVITAI_KEY')
    if civitai_token:
        civitai_token = civitai_token.strip()
        print("✅ Civitai API キーをシークレットから読み込みました")
except Exception:
    pass

if not use_civitai_key:
    print("ℹ️ Civitai APIキーの確認を飛ばしました（H3の利用には不要です）")
elif not civitai_token:
    print("ℹ️ Civitai APIキー未設定です（H3の利用には不要なので、そのままで問題ありません）")

# @markdown ---
# @markdown ## 🔑 Hugging Face API Token（任意・推奨）
# @markdown H3のモデル本体はHugging Faceの公開リポジトリ（`Comfy-Org/MiniMax-H3`）にあり、トークンが無くてもダウンロードできます。
# @markdown ただし、トークンを登録しておくとダウンロード時のレート制限（アクセス制限）にかかりにくくなるため、設定を推奨します。
# @markdown
# @markdown **① トークンの取得**
# @markdown 1. [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) を開く（要ログイン）
# @markdown 2. **＋ Create new token** をクリック（種類は「Read」で十分）
# @markdown 3. 名前を入力して作成 → 表示されたトークンをコピー
# @markdown
# @markdown **② Google Colab シークレットへの登録**
# @markdown 1. 左サイドバーの 🔑 **シークレット**（鍵アイコン）を開く
# @markdown 2. **＋ 新しいシークレットを追加** をクリック
# @markdown 3. 名前に `HF_TOKEN`、値にコピーしたトークンを貼り付けて保存
# @markdown 4. **ノートブックからのアクセス** をオンにする

use_hf_token = False  # @param {type:"boolean"}
# @markdown ↑ オンのときだけ、シークレット `HF_TOKEN` を読みにいきます（オフならシークレットの確認を飛ばし、トークンなしでダウンロードします）。

hf_token = ""

try:
    if not use_hf_token:
        raise RuntimeError("use_hf_token がオフ")
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        hf_token = hf_token.strip()
        print("✅ Hugging Face トークンをシークレットから正常に読み込みました")
    else:
        print("ℹ️ 'HF_TOKEN' は見つかりましたが、中身が空っぽのようです（公開モデルなのでトークンなしでも続行できます）")
except Exception as e:
    print(f"ℹ️ Hugging Face トークンの読み込みをスキップします: {e}")
    print("   （`Comfy-Org/MiniMax-H3` は公開リポジトリのため、トークンが無くてもダウンロード自体は可能です）")
    hf_token = ""

if not hf_token:
    print("ℹ️ Hugging Faceトークンなしで続行します。レート制限にかかった場合は上記手順でHF_TOKENを設定してください。")

# @markdown ---
# @markdown ## 📁 diffusion_models フォルダ
# @markdown Singularity V2 INT8本体。INT8映像VAE・通常Attention・20ステップの比較開始設定です。
diffusion_models_url_1 = "https://huggingface.co/WarmBloodAban/Minimax-h3_Singularity/resolve/1259cf16cd0b655ba691a32fb46a409142762972/Minimax-h3_Singularity_ref2va_v2_pruned_int8.safetensors" # @param {type:"string"}
diffusion_models_url_2 = "" # @param {type:"string"}
diffusion_models_url_3 = "" # @param {type:"string"}
diffusion_models_url_4 = "" # @param {type:"string"}
diffusion_models_url_5 = "" # @param {type:"string"}
diffusion_models_url_6 = "" # @param {type:"string"}
diffusion_models_url_7 = "" # @param {type:"string"}
diffusion_models_url_8 = "" # @param {type:"string"}

# 入力されたURLを結合
diffusion_models_urls = "\n".join([
    url for url in [
        diffusion_models_url_1,
        diffusion_models_url_2,
        diffusion_models_url_3,
        diffusion_models_url_4,
        diffusion_models_url_5,
        diffusion_models_url_6,
        diffusion_models_url_7,
        diffusion_models_url_8
    ] if url.strip()
])

# @markdown ---
# @markdown ## 📁 text_encoders フォルダ
# @markdown MiniMax H3用 テキストエンコーダー（Qwen3-VL-32B 量子化NVFP4-AWQ版・約14.6GiB）。プロンプトと参照画像・参照動画の理解に使用
text_encoders_url_1 = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/7a2065e37f5ff9d3c4e605f164d4cac388eff8e8/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors" # @param {type:"string"}
text_encoders_url_2 = "" # @param {type:"string"}
text_encoders_url_3 = "" # @param {type:"string"}

# 入力されたURLを結合
text_encoders_urls = "\n".join([url for url in [text_encoders_url_1, text_encoders_url_2, text_encoders_url_3] if url.strip()])

# @markdown ---
# @markdown ## 📁 vae フォルダ
# @markdown MiniMax H3用 動画VAEと音声VAE（H3は動画と音声を同時に扱うため両方必須）
vae_url_1 = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/7a2065e37f5ff9d3c4e605f164d4cac388eff8e8/vae/minimax_h3_video_vae_int8_convrot.safetensors" # @param {type:"string"}
vae_url_2 = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/7a2065e37f5ff9d3c4e605f164d4cac388eff8e8/vae/minimax_h3_audio_vae_fp32.safetensors" # @param {type:"string"}
vae_url_3 = "" # @param {type:"string"}

# 入力されたURLを結合
vae_urls = "\n".join([url for url in [vae_url_1, vae_url_2, vae_url_3] if url.strip()])

# @markdown ---
# @markdown ## 📁 clip_vision フォルダ
# @markdown CLIP Visionモデル（画像理解用）
clip_vision_url_1 = "" # @param {type:"string"}
clip_vision_url_2 = "" # @param {type:"string"}
clip_vision_url_3 = "" # @param {type:"string"}

# 入力されたURLを結合
clip_vision_urls = "\n".join([url for url in [clip_vision_url_1, clip_vision_url_2, clip_vision_url_3] if url.strip()])

# @markdown ---
# @markdown ## 📁 loras フォルダ
# @markdown この版は20回計算で比較するためLoRAは空欄。Turboを使う場合は計算設定も変える別検証が必要です。

loras_url_1 = "" # @param {type:"string"}
loras_url_2 = "" # @param {type:"string"}
loras_url_3 = "" # @param {type:"string"}
loras_url_4 = "" # @param {type:"string"}
loras_url_5 = "" # @param {type:"string"}
loras_url_6 = "" # @param {type:"string"}
loras_url_7 = "" # @param {type:"string"}
loras_url_8 = "" # @param {type:"string"}

# URLをまとめる
loras_urls = "\n".join([url for url in [loras_url_1, loras_url_2, loras_url_3, loras_url_4, loras_url_5, loras_url_6, loras_url_7, loras_url_8] if url])

# @markdown ---
# @markdown ## 📁 audio_encoders フォルダ
# @markdown Audio Encoderモデル（音声付き動画生成用）
audio_encoders_url_1 = "" # @param {type:"string"}
audio_encoders_url_2 = "" # @param {type:"string"}
audio_encoders_url_3 = "" # @param {type:"string"}

# URLをまとめる
audio_encoders_urls = "\n".join([url for url in [audio_encoders_url_1, audio_encoders_url_2, audio_encoders_url_3] if url])

# @markdown ---
# @markdown ## 📁 sam2 フォルダ
# @markdown SAM2セグメンテーションモデル
sam2_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 unet フォルダ
# @markdown UNetモデル（Flux等）
unet_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 checkpoints フォルダ
# @markdown チェックポイントモデル
checkpoints_url_1 = "" # @param {type:"string"}
checkpoints_url_2 = "" # @param {type:"string"}
checkpoints_url_3 = "" # @param {type:"string"}
checkpoints_url_4 = "" # @param {type:"string"}
checkpoints_url_5 = "" # @param {type:"string"}

# 入力されたURLを結合
checkpoints_urls = "\n".join([url for url in [checkpoints_url_1, checkpoints_url_2, checkpoints_url_3, checkpoints_url_4, checkpoints_url_5] if url.strip()])

# @markdown ---
# @markdown ## 📁 controlnet フォルダ
# @markdown ControlNetモデル
controlnet_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 clip フォルダ
# @markdown CLIPモデル
clip_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 upscale_models フォルダ
# @markdown アップスケールモデル
upscale_models_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 embeddings フォルダ
# @markdown Embeddingsモデル
embeddings_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 カスタムフォルダ（自由指定）
# @markdown フォルダパスとURLを指定
custom_folder_path = "" # @param {type:"string"}
custom_folder_urls = "" # @param {type:"string"}


# ========================================
# ダウンロード実行
# ========================================

print("=" * 70)
print("🚀 Starting model downloads...")
print("=" * 70)

# 各フォルダへのダウンロード
download_models_to_folder(diffusion_models_urls, "/content/ComfyUI/models/diffusion_models", civitai_token)
download_models_to_folder(text_encoders_urls, "/content/ComfyUI/models/text_encoders", civitai_token)
download_models_to_folder(vae_urls, "/content/ComfyUI/models/vae", civitai_token)
download_models_to_folder(clip_vision_urls, "/content/ComfyUI/models/clip_vision", civitai_token)
download_models_to_folder(loras_urls, "/content/ComfyUI/models/loras", civitai_token)
download_models_to_folder(audio_encoders_urls, "/content/ComfyUI/models/audio_encoders", civitai_token)
download_models_to_folder(sam2_urls, "/content/ComfyUI/models/sam2", civitai_token)
download_models_to_folder(unet_urls, "/content/ComfyUI/models/unet", civitai_token)
download_models_to_folder(checkpoints_urls, "/content/ComfyUI/models/checkpoints", civitai_token)
download_models_to_folder(controlnet_urls, "/content/ComfyUI/models/controlnet", civitai_token)
download_models_to_folder(clip_urls, "/content/ComfyUI/models/clip", civitai_token)
download_models_to_folder(upscale_models_urls, "/content/ComfyUI/models/upscale_models", civitai_token)
download_models_to_folder(embeddings_urls, "/content/ComfyUI/models/embeddings", civitai_token)

# カスタムフォルダ
if custom_folder_path and custom_folder_urls.strip():
    download_models_to_folder(custom_folder_urls, custom_folder_path, civitai_token)

print("=" * 70)
print("✓ All downloads completed!")
print("=" * 70)

# ========================================
# Singularity専用ワークフローの配置
# ========================================
# 20ステップ版のワークフローテンプレート生成（Sol-Attn+EasyCache版の自動生成を含む）は、
# FastH3では設定が噛み合わないため丸ごと差し替えています。
#
# 【重要】配布元のサンプル fasth3_vsa_sample.json は「API形式」で、ノードの座標も
# 接続線の情報も持たないため、ComfyUIの画面で開いても真っ白になります。
# ここでは、それを「UI形式」に変換して動作確認済みのものを埋め込んでいます。
# （2026-09-01にローカルのComfyUIで、実際に画面から開けることを目視確認済み）

import os as _os, json as _json, base64 as _b64

h3_workflow_dir = "/content/ComfyUI/user/default/workflows/Singularity_V2_NewVAE"
_os.makedirs(h3_workflow_dir, exist_ok=True)

_WF_T2V_B64 = (
    "eyJpZCI6IjU5YmVmZTZlLTIzYTUtNDczMS05YTkwLWZjMjkwMzllNzM0MSIsInJldmlzaW9uIjowLCJsYXN0X25vZGVfaWQiOjE2"
    "LCJsYXN0X2xpbmtfaWQiOjE4LCJub2RlcyI6W3siaWQiOjEsInR5cGUiOiJVTkVUTG9hZGVyIiwicG9zIjpbMCwwXSwic2l6ZSI6"
    "WzQwMCw4Ml0sImZsYWdzIjp7fSwib3JkZXIiOjAsIm1vZGUiOjAsImlucHV0cyI6W10sIm91dHB1dHMiOlt7Im5hbWUiOiJNT0RF"
    "TCIsInR5cGUiOiJNT0RFTCIsImxpbmtzIjpbMV19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlVORVRMb2Fk"
    "ZXIifSwid2lkZ2V0c192YWx1ZXMiOlsiTWluaW1heC1oM19TaW5ndWxhcml0eV9yZWYydmFfdjJfcHJ1bmVkX2ludDguc2FmZXRl"
    "bnNvcnMiLCJkZWZhdWx0Il0sInRpdGxlIjoiU2luZ3VsYXJpdHkgVjIgSU5UOCDjg6Ljg4fjg6sifSx7ImlkIjoyLCJ0eXBlIjoi"
    "TWluaU1heEgzU2lnbWFTaGlmdCIsInBvcyI6WzQ2MCwwXSwic2l6ZSI6WzQwMCwxMDhdLCJmbGFncyI6e30sIm9yZGVyIjo3LCJt"
    "b2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJtb2RlbCIsInR5cGUiOiJNT0RFTCIsImxpbmsiOjF9XSwib3V0cHV0cyI6W3sibmFt"
    "ZSI6Ik1PREVMIiwidHlwZSI6Ik1PREVMIiwibGlua3MiOlsyXX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoi"
    "TWluaU1heEgzU2lnbWFTaGlmdCJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WzEyLDNdLCJ0aXRsZSI6IuWLleeUu3NoaWZ0IDEy44O76Z+z"
    "5aOwc2hpZnQgMyJ9LHsiaWQiOjMsInR5cGUiOiJCbG9ja1NwYXJzZUF0dGVudGlvbiIsInBvcyI6WzkyMCwwXSwic2l6ZSI6WzQw"
    "MCwzNTBdLCJmbGFncyI6e30sIm9yZGVyIjo5LCJtb2RlIjo0LCJpbnB1dHMiOlt7Im5hbWUiOiJtb2RlbCIsInR5cGUiOiJNT0RF"
    "TCIsImxpbmsiOjJ9XSwib3V0cHV0cyI6W3sibmFtZSI6Ik1PREVMIiwidHlwZSI6Ik1PREVMIiwibGlua3MiOls1XX1dLCJwcm9w"
    "ZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiQmxvY2tTcGFyc2VBdHRlbnRpb24ifSwid2lkZ2V0c192YWx1ZXMiOlsidnNh"
    "IiwyMCwwLDEsIiIsMCwwLCJleGFjdF9rdl9hbmRfcm93cyIsdHJ1ZV0sIndpZGdldHNfdmFsdWVzX25hbWVkIjp7InNlbGVjdGlv"
    "bi5rZWVwX3BlcmNlbnQiOjIwLCJlbmRfcGVyY2VudCI6MSwiZXh0cmFfdG9rZW5zIjowLCJzdGFydF9wZXJjZW50IjowLCJkZW5z"
    "ZV9ibG9ja3MiOiIiLCJzZWxlY3Rpb24iOiJ2c2EiLCJ2ZXJib3NlIjp0cnVlLCJzaW5rX2NvbmRpdGlvbmluZyI6ImV4YWN0X2t2"
    "X2FuZF9yb3dzIiwibWluX3Rva2VucyI6MH0sInRpdGxlIjoi54Sh5Yq577yI44OQ44Kk44OR44K577yJOiDpgJrluLhBdHRlbnRp"
    "b27jgafnlJ/miJAifSx7ImlkIjo0LCJ0eXBlIjoiQ0xJUExvYWRlciIsInBvcyI6WzAsMjYwXSwic2l6ZSI6WzQwMCwxMDhdLCJm"
    "bGFncyI6e30sIm9yZGVyIjoxLCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiQ0xJUCIsInR5cGUiOiJD"
    "TElQIiwibGlua3MiOlszXX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiQ0xJUExvYWRlciJ9LCJ3aWRnZXRz"
    "X3ZhbHVlcyI6WyJxd2VuM3ZsXzMyYl9taW5pbWF4X2gzX252ZnA0X2F3cS5zYWZldGVuc29ycyIsIm1pbmltYXgiLCJkZWZhdWx0"
    "Il19LHsiaWQiOjUsInR5cGUiOiJWQUVMb2FkZXIiLCJwb3MiOlswLDUyMF0sInNpemUiOls0MDAsNjBdLCJmbGFncyI6e30sIm9y"
    "ZGVyIjoyLCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiVkFFIiwidHlwZSI6IlZBRSIsImxpbmtzIjpb"
    "NCwxM119XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlZBRUxvYWRlciJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJt"
    "aW5pbWF4X2gzX3ZpZGVvX3ZhZV9pbnQ4X2NvbnZyb3Quc2FmZXRlbnNvcnMiXX0seyJpZCI6NiwidHlwZSI6IlZBRUxvYWRlciIs"
    "InBvcyI6WzAsNzgwXSwic2l6ZSI6WzQwMCw2MF0sImZsYWdzIjp7fSwib3JkZXIiOjMsIm1vZGUiOjAsImlucHV0cyI6W10sIm91"
    "dHB1dHMiOlt7Im5hbWUiOiJWQUUiLCJ0eXBlIjoiVkFFIiwibGlua3MiOlsxNV19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUg"
    "Zm9yIFMmUiI6IlZBRUxvYWRlciJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJtaW5pbWF4X2gzX2F1ZGlvX3ZhZV9mcDMyLnNhZmV0ZW5z"
    "b3JzIl19LHsiaWQiOjcsInR5cGUiOiJNaW5pTWF4SDNJbWFnZVRvVmlkZW8iLCJwb3MiOls0NjAsMjYwXSwic2l6ZSI6WzQwMCwy"
    "MzhdLCJmbGFncyI6e30sIm9yZGVyIjo4LCJtb2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJjbGlwIiwidHlwZSI6IkNMSVAiLCJs"
    "aW5rIjozfSx7Im5hbWUiOiJ2YWUiLCJ0eXBlIjoiVkFFIiwibGluayI6NH0seyJuYW1lIjoiZmlyc3RfZnJhbWUiLCJ0eXBlIjoi"
    "SU1BR0UiLCJsaW5rIjpudWxsLCJzaGFwZSI6N30seyJuYW1lIjoibGFzdF9mcmFtZSIsInR5cGUiOiJJTUFHRSIsImxpbmsiOm51"
    "bGwsInNoYXBlIjo3fV0sIm91dHB1dHMiOlt7Im5hbWUiOiJwb3NpdGl2ZSIsInR5cGUiOiJDT05ESVRJT05JTkciLCJsaW5rcyI6"
    "WzZdfSx7Im5hbWUiOiJMQVRFTlQiLCJ0eXBlIjoiTEFURU5UIiwibGlua3MiOlsxMV19XSwicHJvcGVydGllcyI6eyJOb2RlIG5h"
    "bWUgZm9yIFMmUiI6Ik1pbmlNYXhIM0ltYWdlVG9WaWRlbyJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJpbnRlZ3JhdGVkX211bHRpbW9k"
    "YWxfZGVzY3JpcHRpb246IFtTaG90IDFdIDJELWFuaW1hdGVkLCBhbmltZSBzdHlsZSwgYSBtZWRpdW0gY2xvc2UtdXAgc2hvdCBm"
    "cmFtZXMgYSBjaGVlcmZ1bCB5b3VuZyB3b21hbiB3aXRoIGRhcmsgaGFpciB0aWVkIHVwIGluIGEgbmVhdCBidW4sIHNpZGUgYmFu"
    "Z3MgZnJhbWluZyBoZXIgZGVsaWNhdGUgZmFjZSwgYW5kIGRhcmsgcHVycGxlIGV5ZXMgd2VhcmluZyBzbWFsbCBwZWFybCBlYXJy"
    "aW5ncy4gU2hlIHdlYXJzIGEgbGlnaHQgZ3JheSB0dXJ0bGVuZWNrIGtuaXQgdG9wIHVuZGVyIGEgYmxhY2sgc3BhZ2hldHRpLXN0"
    "cmFwIG1pbmkgZHJlc3MuIFN0YW5kaW5nIGluIGEgY2xlYW4gbW9kZXJuIHJvb20gd2l0aCB3aGl0ZSB3YWxscyBhbmQgYSBkYXJr"
    "IGRvb3JmcmFtZSwgc2hlIHRpbHRzIGhlciBoZWFkIHNsaWdodGx5IHdpdGggYSBicmlnaHQgc21pbGUsIGxpZ2h0bHkgcmVzdGlu"
    "ZyBoZXIgcmlnaHQgZmluZ2VydGlwcyBuZWFyIGhlciBjb2xsYXJib25lLiBUaGUgY2FtZXJhIHB1c2hlcyBpbiB3aXRoIHNtYWxs"
    "IGFtcGxpdHVkZSBhdCBzbG93IHNwZWVkIGFzIHRoZSB5b3VuZyB3b21hbiB3aXRoIGEgc3dlZXQsIGNsZWFyIHZvaWNlIChTMSkg"
    "Z2VudGx5IHNheXM6IDxkPltKYXBhbmVzZV0g44GT44KT44Gr44Gh44Gv77yB5LuK5pel44KC5LiA5pel6aCR5by144KN44GG44Gt"
    "44CCPC9kPlxuXG5vdmVyYWxsX3NvdW5kc2NhcGU6IFF1aWV0IGluZG9vciByb29tIHRvbmUsIGEgc29mdCBydXN0bGUgb2Yga25p"
    "dCBmYWJyaWMgYXMgc2hlIG1vdmVzIGhlciBzaG91bGRlciwgYW5kIHRoZSBjbGVhciBzcG9rZW4gdm9pY2UgaW4gYSBuYXR1cmFs"
    "IHJvb20gYWNvdXN0aWMuXG5cbm5vbl9kaWVnZXRpY19tdXNpYzogQSBnZW50bGUsIHVwbGlmdGluZyBhY291c3RpYyBndWl0YXIg"
    "cGF0dGVybiB3aXRoIHdhcm0gcGlhbm8gbm90ZXMgYXQgYSBtb2RlcmF0ZSB0ZW1wbywgZmFkaW5nIHNvZnRseSBhdCB0aGUgZW5k"
    "LiIsNTQ0LDgzMiwxMjRdfSx7ImlkIjo4LCJ0eXBlIjoiUmFuZG9tTm9pc2UiLCJwb3MiOlswLDEwNDBdLCJzaXplIjpbNDAwLDYw"
    "XSwiZmxhZ3MiOnt9LCJvcmRlciI6NCwibW9kZSI6MCwiaW5wdXRzIjpbXSwib3V0cHV0cyI6W3sibmFtZSI6Ik5PSVNFIiwidHlw"
    "ZSI6Ik5PSVNFIiwibGlua3MiOls3XX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiUmFuZG9tTm9pc2UifSwi"
    "d2lkZ2V0c192YWx1ZXMiOls0Ml19LHsiaWQiOjksInR5cGUiOiJCYXNpY0d1aWRlciIsInBvcyI6WzEzODAsMF0sInNpemUiOls0"
    "MDAsODJdLCJmbGFncyI6e30sIm9yZGVyIjoxMCwibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoibW9kZWwiLCJ0eXBlIjoiTU9E"
    "RUwiLCJsaW5rIjo1fSx7Im5hbWUiOiJjb25kaXRpb25pbmciLCJ0eXBlIjoiQ09ORElUSU9OSU5HIiwibGluayI6Nn1dLCJvdXRw"
    "dXRzIjpbeyJuYW1lIjoiR1VJREVSIiwidHlwZSI6IkdVSURFUiIsImxpbmtzIjpbOF19XSwicHJvcGVydGllcyI6eyJOb2RlIG5h"
    "bWUgZm9yIFMmUiI6IkJhc2ljR3VpZGVyIn0sIndpZGdldHNfdmFsdWVzIjpbXX0seyJpZCI6MTAsInR5cGUiOiJLU2FtcGxlclNl"
    "bGVjdCIsInBvcyI6WzAsMTMwMF0sInNpemUiOls0MDAsNjBdLCJmbGFncyI6e30sIm9yZGVyIjo1LCJtb2RlIjowLCJpbnB1dHMi"
    "OltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiU0FNUExFUiIsInR5cGUiOiJTQU1QTEVSIiwibGlua3MiOls5XX1dLCJwcm9wZXJ0aWVz"
    "Ijp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiS1NhbXBsZXJTZWxlY3QifSwid2lkZ2V0c192YWx1ZXMiOlsiZXVsZXIiXX0seyJpZCI6"
    "MTEsInR5cGUiOiJNYW51YWxTaWdtYXMiLCJwb3MiOlswLDE1NjBdLCJzaXplIjpbNDAwLDYwXSwiZmxhZ3MiOnt9LCJvcmRlciI6"
    "NiwibW9kZSI6MCwiaW5wdXRzIjpbXSwib3V0cHV0cyI6W3sibmFtZSI6IlNJR01BUyIsInR5cGUiOiJTSUdNQVMiLCJsaW5rcyI6"
    "WzEwXX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiTWFudWFsU2lnbWFzIn0sIndpZGdldHNfdmFsdWVzIjpb"
    "IjAuOTk5ODk5OTA5OSwgMC45OTU2MzMxODc4LCAwLjk5MDgyNTY4ODEsIDAuOTg1NTA3MjQ2NCwgMC45Nzk1OTE4MzY3LCAwLjk3"
    "Mjk3Mjk3MzAsIDAuOTY1NTE3MjQxNCwgMC45NTcwNTUyMTQ3LCAwLjk0NzM2ODQyMTEsIDAuOTM2MTcwMjEyOCwgMC45MjMwNzY5"
    "MjMxLCAwLjkwNzU2MzAyNTIsIDAuODg4ODg4ODg4OSwgMC44NjU5NzkzODE0LCAwLjgzNzIwOTMwMjMsIDAuODAwMDAwMDAwMCwg"
    "MC43NTAwMDAwMDAwLCAwLjY3OTI0NTI4MzAsIDAuNTcxNDI4NTcxNCwgMC4zODcwOTY3NzQyLCAwIl0sInRpdGxlIjoiU2luZ3Vs"
    "YXJpdHkgVjIgMjDjgrnjg4bjg4Pjg5fvvIjmr5TovIPplovlp4voqK3lrprvvIkifSx7ImlkIjoxMiwidHlwZSI6IlNhbXBsZXJD"
    "dXN0b21BZHZhbmNlZCIsInBvcyI6WzE4NDAsMF0sInNpemUiOls0MDAsMTYwXSwiZmxhZ3MiOnt9LCJvcmRlciI6MTEsIm1vZGUi"
    "OjAsImlucHV0cyI6W3sibmFtZSI6Im5vaXNlIiwidHlwZSI6Ik5PSVNFIiwibGluayI6N30seyJuYW1lIjoiZ3VpZGVyIiwidHlw"
    "ZSI6IkdVSURFUiIsImxpbmsiOjh9LHsibmFtZSI6InNhbXBsZXIiLCJ0eXBlIjoiU0FNUExFUiIsImxpbmsiOjl9LHsibmFtZSI6"
    "InNpZ21hcyIsInR5cGUiOiJTSUdNQVMiLCJsaW5rIjoxMH0seyJuYW1lIjoibGF0ZW50X2ltYWdlIiwidHlwZSI6IkxBVEVOVCIs"
    "ImxpbmsiOjExfV0sIm91dHB1dHMiOlt7Im5hbWUiOiJvdXRwdXQiLCJ0eXBlIjoiTEFURU5UIiwibGlua3MiOlsxMiwxNF19LHsi"
    "bmFtZSI6ImRlbm9pc2VkX291dHB1dCIsInR5cGUiOiJMQVRFTlQiLCJsaW5rcyI6bnVsbH1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUg"
    "bmFtZSBmb3IgUyZSIjoiU2FtcGxlckN1c3RvbUFkdmFuY2VkIn0sIndpZGdldHNfdmFsdWVzIjpbXX0seyJpZCI6MTMsInR5cGUi"
    "OiJWQUVEZWNvZGUiLCJwb3MiOlsyMzAwLDBdLCJzaXplIjpbNDAwLDgyXSwiZmxhZ3MiOnt9LCJvcmRlciI6MTIsIm1vZGUiOjAs"
    "ImlucHV0cyI6W3sibmFtZSI6InNhbXBsZXMiLCJ0eXBlIjoiTEFURU5UIiwibGluayI6MTJ9LHsibmFtZSI6InZhZSIsInR5cGUi"
    "OiJWQUUiLCJsaW5rIjoxM31dLCJvdXRwdXRzIjpbeyJuYW1lIjoiSU1BR0UiLCJ0eXBlIjoiSU1BR0UiLCJsaW5rcyI6WzE2XX1d"
    "LCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiVkFFRGVjb2RlIn0sIndpZGdldHNfdmFsdWVzIjpbXX0seyJpZCI6"
    "MTQsInR5cGUiOiJWQUVEZWNvZGVBdWRpbyIsInBvcyI6WzIzMDAsMjYwXSwic2l6ZSI6WzQwMCw4Ml0sImZsYWdzIjp7fSwib3Jk"
    "ZXIiOjEzLCJtb2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJzYW1wbGVzIiwidHlwZSI6IkxBVEVOVCIsImxpbmsiOjE0fSx7Im5h"
    "bWUiOiJ2YWUiLCJ0eXBlIjoiVkFFIiwibGluayI6MTV9XSwib3V0cHV0cyI6W3sibmFtZSI6IkFVRElPIiwidHlwZSI6IkFVRElP"
    "IiwibGlua3MiOlsxN119XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlZBRURlY29kZUF1ZGlvIn0sIndpZGdl"
    "dHNfdmFsdWVzIjpbXX0seyJpZCI6MTUsInR5cGUiOiJDcmVhdGVWaWRlbyIsInBvcyI6WzI3NjAsMF0sInNpemUiOls0MDAsMTYw"
    "XSwiZmxhZ3MiOnt9LCJvcmRlciI6MTQsIm1vZGUiOjAsImlucHV0cyI6W3sibmFtZSI6ImltYWdlcyIsInR5cGUiOiJJTUFHRSIs"
    "ImxpbmsiOjE2fSx7Im5hbWUiOiJhdWRpbyIsInR5cGUiOiJBVURJTyIsImxpbmsiOjE3LCJzaGFwZSI6N31dLCJvdXRwdXRzIjpb"
    "eyJuYW1lIjoiVklERU8iLCJ0eXBlIjoiVklERU8iLCJsaW5rcyI6WzE4XX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3Ig"
    "UyZSIjoiQ3JlYXRlVmlkZW8ifSwid2lkZ2V0c192YWx1ZXMiOlsyNCw4LCJzUkdCIl19LHsiaWQiOjE2LCJ0eXBlIjoiU2F2ZVZp"
    "ZGVvIiwicG9zIjpbMzIyMCwwXSwic2l6ZSI6WzQwMCwxMzRdLCJmbGFncyI6e30sIm9yZGVyIjoxNSwibW9kZSI6MCwiaW5wdXRz"
    "IjpbeyJuYW1lIjoidmlkZW8iLCJ0eXBlIjoiVklERU8iLCJsaW5rIjoxOH1dLCJvdXRwdXRzIjpbeyJuYW1lIjoidmlkZW8iLCJ0"
    "eXBlIjoiVklERU8iLCJsaW5rcyI6bnVsbH1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiU2F2ZVZpZGVvIn0s"
    "IndpZGdldHNfdmFsdWVzIjpbIlNpbmd1bGFyaXR5X1YyL3Npbmd1bGFyaXR5X3YyIiwiYXV0byIsImF1dG8iXX1dLCJsaW5rcyI6"
    "W1sxLDEsMCwyLDAsIk1PREVMIl0sWzIsMiwwLDMsMCwiTU9ERUwiXSxbMyw0LDAsNywwLCJDTElQIl0sWzQsNSwwLDcsMSwiVkFF"
    "Il0sWzUsMywwLDksMCwiTU9ERUwiXSxbNiw3LDAsOSwxLCJDT05ESVRJT05JTkciXSxbNyw4LDAsMTIsMCwiTk9JU0UiXSxbOCw5"
    "LDAsMTIsMSwiR1VJREVSIl0sWzksMTAsMCwxMiwyLCJTQU1QTEVSIl0sWzEwLDExLDAsMTIsMywiU0lHTUFTIl0sWzExLDcsMSwx"
    "Miw0LCJMQVRFTlQiXSxbMTIsMTIsMCwxMywwLCJMQVRFTlQiXSxbMTMsNSwwLDEzLDEsIlZBRSJdLFsxNCwxMiwwLDE0LDAsIkxB"
    "VEVOVCJdLFsxNSw2LDAsMTQsMSwiVkFFIl0sWzE2LDEzLDAsMTUsMCwiSU1BR0UiXSxbMTcsMTQsMCwxNSwxLCJBVURJTyJdLFsx"
    "OCwxNSwwLDE2LDAsIlZJREVPIl1dLCJncm91cHMiOltdLCJjb25maWciOnt9LCJleHRyYSI6eyJkcyI6eyJzY2FsZSI6MC43LCJv"
    "ZmZzZXQiOlswLDBdfX0sInZlcnNpb24iOjAuNH0="
)
_WF_I2V_B64 = (
    "eyJpZCI6Ijc3ZDE4YjFmLWVmMTItNDViZS1hNTA2LTJhNmM2MDk4Zjc1NSIsInJldmlzaW9uIjowLCJsYXN0X25vZGVfaWQiOjE3"
    "LCJsYXN0X2xpbmtfaWQiOjE5LCJub2RlcyI6W3siaWQiOjEsInR5cGUiOiJVTkVUTG9hZGVyIiwicG9zIjpbMCwwXSwic2l6ZSI6"
    "WzQwMCw4Ml0sImZsYWdzIjp7fSwib3JkZXIiOjAsIm1vZGUiOjAsImlucHV0cyI6W10sIm91dHB1dHMiOlt7Im5hbWUiOiJNT0RF"
    "TCIsInR5cGUiOiJNT0RFTCIsImxpbmtzIjpbMV19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlVORVRMb2Fk"
    "ZXIifSwid2lkZ2V0c192YWx1ZXMiOlsiTWluaW1heC1oM19TaW5ndWxhcml0eV9yZWYydmFfdjJfcHJ1bmVkX2ludDguc2FmZXRl"
    "bnNvcnMiLCJkZWZhdWx0Il0sInRpdGxlIjoiU2luZ3VsYXJpdHkgVjIgSU5UOCDjg6Ljg4fjg6sifSx7ImlkIjoyLCJ0eXBlIjoi"
    "TWluaU1heEgzU2lnbWFTaGlmdCIsInBvcyI6WzQ2MCwwXSwic2l6ZSI6WzQwMCwxMDhdLCJmbGFncyI6e30sIm9yZGVyIjo4LCJt"
    "b2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJtb2RlbCIsInR5cGUiOiJNT0RFTCIsImxpbmsiOjF9XSwib3V0cHV0cyI6W3sibmFt"
    "ZSI6Ik1PREVMIiwidHlwZSI6Ik1PREVMIiwibGlua3MiOlsyXX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoi"
    "TWluaU1heEgzU2lnbWFTaGlmdCJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WzEyLDNdLCJ0aXRsZSI6IuWLleeUu3NoaWZ0IDEy44O76Z+z"
    "5aOwc2hpZnQgMyJ9LHsiaWQiOjMsInR5cGUiOiJCbG9ja1NwYXJzZUF0dGVudGlvbiIsInBvcyI6WzkyMCwwXSwic2l6ZSI6WzQw"
    "MCwzNTBdLCJmbGFncyI6e30sIm9yZGVyIjoxMCwibW9kZSI6NCwiaW5wdXRzIjpbeyJuYW1lIjoibW9kZWwiLCJ0eXBlIjoiTU9E"
    "RUwiLCJsaW5rIjoyfV0sIm91dHB1dHMiOlt7Im5hbWUiOiJNT0RFTCIsInR5cGUiOiJNT0RFTCIsImxpbmtzIjpbNl19XSwicHJv"
    "cGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IkJsb2NrU3BhcnNlQXR0ZW50aW9uIn0sIndpZGdldHNfdmFsdWVzIjpbInZz"
    "YSIsMjAsMCwxLCIiLDAsMCwiZXhhY3Rfa3ZfYW5kX3Jvd3MiLHRydWVdLCJ3aWRnZXRzX3ZhbHVlc19uYW1lZCI6eyJzZWxlY3Rp"
    "b24ua2VlcF9wZXJjZW50IjoyMCwiZW5kX3BlcmNlbnQiOjEsImV4dHJhX3Rva2VucyI6MCwic3RhcnRfcGVyY2VudCI6MCwiZGVu"
    "c2VfYmxvY2tzIjoiIiwic2VsZWN0aW9uIjoidnNhIiwidmVyYm9zZSI6dHJ1ZSwic2lua19jb25kaXRpb25pbmciOiJleGFjdF9r"
    "dl9hbmRfcm93cyIsIm1pbl90b2tlbnMiOjB9LCJ0aXRsZSI6IueEoeWKue+8iOODkOOCpOODkeOCue+8iTog6YCa5bi4QXR0ZW50"
    "aW9u44Gn55Sf5oiQIn0seyJpZCI6NCwidHlwZSI6IkNMSVBMb2FkZXIiLCJwb3MiOlswLDI2MF0sInNpemUiOls0MDAsMTA4XSwi"
    "ZmxhZ3MiOnt9LCJvcmRlciI6MSwibW9kZSI6MCwiaW5wdXRzIjpbXSwib3V0cHV0cyI6W3sibmFtZSI6IkNMSVAiLCJ0eXBlIjoi"
    "Q0xJUCIsImxpbmtzIjpbM119XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IkNMSVBMb2FkZXIifSwid2lkZ2V0"
    "c192YWx1ZXMiOlsicXdlbjN2bF8zMmJfbWluaW1heF9oM19udmZwNF9hd3Euc2FmZXRlbnNvcnMiLCJtaW5pbWF4IiwiZGVmYXVs"
    "dCJdfSx7ImlkIjo1LCJ0eXBlIjoiVkFFTG9hZGVyIiwicG9zIjpbMCw1MjBdLCJzaXplIjpbNDAwLDYwXSwiZmxhZ3MiOnt9LCJv"
    "cmRlciI6MiwibW9kZSI6MCwiaW5wdXRzIjpbXSwib3V0cHV0cyI6W3sibmFtZSI6IlZBRSIsInR5cGUiOiJWQUUiLCJsaW5rcyI6"
    "WzQsMTRdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJWQUVMb2FkZXIifSwid2lkZ2V0c192YWx1ZXMiOlsi"
    "bWluaW1heF9oM192aWRlb192YWVfaW50OF9jb252cm90LnNhZmV0ZW5zb3JzIl19LHsiaWQiOjYsInR5cGUiOiJWQUVMb2FkZXIi"
    "LCJwb3MiOlswLDc4MF0sInNpemUiOls0MDAsNjBdLCJmbGFncyI6e30sIm9yZGVyIjozLCJtb2RlIjowLCJpbnB1dHMiOltdLCJv"
    "dXRwdXRzIjpbeyJuYW1lIjoiVkFFIiwidHlwZSI6IlZBRSIsImxpbmtzIjpbMTZdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1l"
    "IGZvciBTJlIiOiJWQUVMb2FkZXIifSwid2lkZ2V0c192YWx1ZXMiOlsibWluaW1heF9oM19hdWRpb192YWVfZnAzMi5zYWZldGVu"
    "c29ycyJdfSx7ImlkIjo3LCJ0eXBlIjoiTWluaU1heEgzSW1hZ2VUb1ZpZGVvIiwicG9zIjpbNDYwLDI2MF0sInNpemUiOls0MDAs"
    "MjM4XSwiZmxhZ3MiOnt9LCJvcmRlciI6OSwibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoiY2xpcCIsInR5cGUiOiJDTElQIiwi"
    "bGluayI6M30seyJuYW1lIjoidmFlIiwidHlwZSI6IlZBRSIsImxpbmsiOjR9LHsibmFtZSI6ImZpcnN0X2ZyYW1lIiwidHlwZSI6"
    "IklNQUdFIiwibGluayI6NSwic2hhcGUiOjd9LHsibmFtZSI6Imxhc3RfZnJhbWUiLCJ0eXBlIjoiSU1BR0UiLCJsaW5rIjpudWxs"
    "LCJzaGFwZSI6N31dLCJvdXRwdXRzIjpbeyJuYW1lIjoicG9zaXRpdmUiLCJ0eXBlIjoiQ09ORElUSU9OSU5HIiwibGlua3MiOls3"
    "XX0seyJuYW1lIjoiTEFURU5UIiwidHlwZSI6IkxBVEVOVCIsImxpbmtzIjpbMTJdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1l"
    "IGZvciBTJlIiOiJNaW5pTWF4SDNJbWFnZVRvVmlkZW8ifSwid2lkZ2V0c192YWx1ZXMiOlsiaW50ZWdyYXRlZF9tdWx0aW1vZGFs"
    "X2Rlc2NyaXB0aW9uOiBbU2hvdCAxXSAyRC1hbmltYXRlZCwgYW5pbWUgc3R5bGUsIGEgbWVkaXVtIGNsb3NlLXVwIHNob3QgZnJh"
    "bWVzIGEgY2hlZXJmdWwgeW91bmcgd29tYW4gd2l0aCBkYXJrIGhhaXIgdGllZCB1cCBpbiBhIG5lYXQgYnVuLCBzaWRlIGJhbmdz"
    "IGZyYW1pbmcgaGVyIGRlbGljYXRlIGZhY2UsIGFuZCBkYXJrIHB1cnBsZSBleWVzIHdlYXJpbmcgc21hbGwgcGVhcmwgZWFycmlu"
    "Z3MuIFNoZSB3ZWFycyBhIGxpZ2h0IGdyYXkgdHVydGxlbmVjayBrbml0IHRvcCB1bmRlciBhIGJsYWNrIHNwYWdoZXR0aS1zdHJh"
    "cCBtaW5pIGRyZXNzLiBTdGFuZGluZyBpbiBhIGNsZWFuIG1vZGVybiByb29tIHdpdGggd2hpdGUgd2FsbHMgYW5kIGEgZGFyayBk"
    "b29yZnJhbWUsIHNoZSB0aWx0cyBoZXIgaGVhZCBzbGlnaHRseSB3aXRoIGEgYnJpZ2h0IHNtaWxlLCBsaWdodGx5IHJlc3Rpbmcg"
    "aGVyIHJpZ2h0IGZpbmdlcnRpcHMgbmVhciBoZXIgY29sbGFyYm9uZS4gVGhlIGNhbWVyYSBwdXNoZXMgaW4gd2l0aCBzbWFsbCBh"
    "bXBsaXR1ZGUgYXQgc2xvdyBzcGVlZCBhcyB0aGUgeW91bmcgd29tYW4gd2l0aCBhIHN3ZWV0LCBjbGVhciB2b2ljZSAoUzEpIGdl"
    "bnRseSBzYXlzOiA8ZD5bSmFwYW5lc2VdIOOBk+OCk+OBq+OBoeOBr++8geS7iuaXpeOCguS4gOaXpemgkeW8teOCjeOBhuOBreOA"
    "gjwvZD5cblxub3ZlcmFsbF9zb3VuZHNjYXBlOiBRdWlldCBpbmRvb3Igcm9vbSB0b25lLCBhIHNvZnQgcnVzdGxlIG9mIGtuaXQg"
    "ZmFicmljIGFzIHNoZSBtb3ZlcyBoZXIgc2hvdWxkZXIsIGFuZCB0aGUgY2xlYXIgc3Bva2VuIHZvaWNlIGluIGEgbmF0dXJhbCBy"
    "b29tIGFjb3VzdGljLlxuXG5ub25fZGllZ2V0aWNfbXVzaWM6IEEgZ2VudGxlLCB1cGxpZnRpbmcgYWNvdXN0aWMgZ3VpdGFyIHBh"
    "dHRlcm4gd2l0aCB3YXJtIHBpYW5vIG5vdGVzIGF0IGEgbW9kZXJhdGUgdGVtcG8sIGZhZGluZyBzb2Z0bHkgYXQgdGhlIGVuZC4i"
    "LDU0NCw4MzIsMTI0XX0seyJpZCI6OCwidHlwZSI6IlJhbmRvbU5vaXNlIiwicG9zIjpbMCwxMDQwXSwic2l6ZSI6WzQwMCw2MF0s"
    "ImZsYWdzIjp7fSwib3JkZXIiOjQsIm1vZGUiOjAsImlucHV0cyI6W10sIm91dHB1dHMiOlt7Im5hbWUiOiJOT0lTRSIsInR5cGUi"
    "OiJOT0lTRSIsImxpbmtzIjpbOF19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlJhbmRvbU5vaXNlIn0sIndp"
    "ZGdldHNfdmFsdWVzIjpbNDJdfSx7ImlkIjo5LCJ0eXBlIjoiQmFzaWNHdWlkZXIiLCJwb3MiOlsxMzgwLDBdLCJzaXplIjpbNDAw"
    "LDgyXSwiZmxhZ3MiOnt9LCJvcmRlciI6MTEsIm1vZGUiOjAsImlucHV0cyI6W3sibmFtZSI6Im1vZGVsIiwidHlwZSI6Ik1PREVM"
    "IiwibGluayI6Nn0seyJuYW1lIjoiY29uZGl0aW9uaW5nIiwidHlwZSI6IkNPTkRJVElPTklORyIsImxpbmsiOjd9XSwib3V0cHV0"
    "cyI6W3sibmFtZSI6IkdVSURFUiIsInR5cGUiOiJHVUlERVIiLCJsaW5rcyI6WzldfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1l"
    "IGZvciBTJlIiOiJCYXNpY0d1aWRlciJ9LCJ3aWRnZXRzX3ZhbHVlcyI6W119LHsiaWQiOjEwLCJ0eXBlIjoiS1NhbXBsZXJTZWxl"
    "Y3QiLCJwb3MiOlswLDEzMDBdLCJzaXplIjpbNDAwLDYwXSwiZmxhZ3MiOnt9LCJvcmRlciI6NSwibW9kZSI6MCwiaW5wdXRzIjpb"
    "XSwib3V0cHV0cyI6W3sibmFtZSI6IlNBTVBMRVIiLCJ0eXBlIjoiU0FNUExFUiIsImxpbmtzIjpbMTBdfV0sInByb3BlcnRpZXMi"
    "OnsiTm9kZSBuYW1lIGZvciBTJlIiOiJLU2FtcGxlclNlbGVjdCJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJldWxlciJdfSx7ImlkIjox"
    "MSwidHlwZSI6Ik1hbnVhbFNpZ21hcyIsInBvcyI6WzAsMTU2MF0sInNpemUiOls0MDAsNjBdLCJmbGFncyI6e30sIm9yZGVyIjo2"
    "LCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiU0lHTUFTIiwidHlwZSI6IlNJR01BUyIsImxpbmtzIjpb"
    "MTFdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJNYW51YWxTaWdtYXMifSwid2lkZ2V0c192YWx1ZXMiOlsi"
    "MC45OTk4OTk5MDk5LCAwLjk5NTYzMzE4NzgsIDAuOTkwODI1Njg4MSwgMC45ODU1MDcyNDY0LCAwLjk3OTU5MTgzNjcsIDAuOTcy"
    "OTcyOTczMCwgMC45NjU1MTcyNDE0LCAwLjk1NzA1NTIxNDcsIDAuOTQ3MzY4NDIxMSwgMC45MzYxNzAyMTI4LCAwLjkyMzA3Njky"
    "MzEsIDAuOTA3NTYzMDI1MiwgMC44ODg4ODg4ODg5LCAwLjg2NTk3OTM4MTQsIDAuODM3MjA5MzAyMywgMC44MDAwMDAwMDAwLCAw"
    "Ljc1MDAwMDAwMDAsIDAuNjc5MjQ1MjgzMCwgMC41NzE0Mjg1NzE0LCAwLjM4NzA5Njc3NDIsIDAiXSwidGl0bGUiOiJTaW5ndWxh"
    "cml0eSBWMiAyMOOCueODhuODg+ODl++8iOavlOi8g+mWi+Wni+ioreWumu+8iSJ9LHsiaWQiOjEyLCJ0eXBlIjoiU2FtcGxlckN1"
    "c3RvbUFkdmFuY2VkIiwicG9zIjpbMTg0MCwwXSwic2l6ZSI6WzQwMCwxNjBdLCJmbGFncyI6e30sIm9yZGVyIjoxMiwibW9kZSI6"
    "MCwiaW5wdXRzIjpbeyJuYW1lIjoibm9pc2UiLCJ0eXBlIjoiTk9JU0UiLCJsaW5rIjo4fSx7Im5hbWUiOiJndWlkZXIiLCJ0eXBl"
    "IjoiR1VJREVSIiwibGluayI6OX0seyJuYW1lIjoic2FtcGxlciIsInR5cGUiOiJTQU1QTEVSIiwibGluayI6MTB9LHsibmFtZSI6"
    "InNpZ21hcyIsInR5cGUiOiJTSUdNQVMiLCJsaW5rIjoxMX0seyJuYW1lIjoibGF0ZW50X2ltYWdlIiwidHlwZSI6IkxBVEVOVCIs"
    "ImxpbmsiOjEyfV0sIm91dHB1dHMiOlt7Im5hbWUiOiJvdXRwdXQiLCJ0eXBlIjoiTEFURU5UIiwibGlua3MiOlsxMywxNV19LHsi"
    "bmFtZSI6ImRlbm9pc2VkX291dHB1dCIsInR5cGUiOiJMQVRFTlQiLCJsaW5rcyI6bnVsbH1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUg"
    "bmFtZSBmb3IgUyZSIjoiU2FtcGxlckN1c3RvbUFkdmFuY2VkIn0sIndpZGdldHNfdmFsdWVzIjpbXX0seyJpZCI6MTMsInR5cGUi"
    "OiJWQUVEZWNvZGUiLCJwb3MiOlsyMzAwLDBdLCJzaXplIjpbNDAwLDgyXSwiZmxhZ3MiOnt9LCJvcmRlciI6MTMsIm1vZGUiOjAs"
    "ImlucHV0cyI6W3sibmFtZSI6InNhbXBsZXMiLCJ0eXBlIjoiTEFURU5UIiwibGluayI6MTN9LHsibmFtZSI6InZhZSIsInR5cGUi"
    "OiJWQUUiLCJsaW5rIjoxNH1dLCJvdXRwdXRzIjpbeyJuYW1lIjoiSU1BR0UiLCJ0eXBlIjoiSU1BR0UiLCJsaW5rcyI6WzE3XX1d"
    "LCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiVkFFRGVjb2RlIn0sIndpZGdldHNfdmFsdWVzIjpbXX0seyJpZCI6"
    "MTQsInR5cGUiOiJWQUVEZWNvZGVBdWRpbyIsInBvcyI6WzIzMDAsMjYwXSwic2l6ZSI6WzQwMCw4Ml0sImZsYWdzIjp7fSwib3Jk"
    "ZXIiOjE0LCJtb2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJzYW1wbGVzIiwidHlwZSI6IkxBVEVOVCIsImxpbmsiOjE1fSx7Im5h"
    "bWUiOiJ2YWUiLCJ0eXBlIjoiVkFFIiwibGluayI6MTZ9XSwib3V0cHV0cyI6W3sibmFtZSI6IkFVRElPIiwidHlwZSI6IkFVRElP"
    "IiwibGlua3MiOlsxOF19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlZBRURlY29kZUF1ZGlvIn0sIndpZGdl"
    "dHNfdmFsdWVzIjpbXX0seyJpZCI6MTUsInR5cGUiOiJDcmVhdGVWaWRlbyIsInBvcyI6WzI3NjAsMF0sInNpemUiOls0MDAsMTYw"
    "XSwiZmxhZ3MiOnt9LCJvcmRlciI6MTUsIm1vZGUiOjAsImlucHV0cyI6W3sibmFtZSI6ImltYWdlcyIsInR5cGUiOiJJTUFHRSIs"
    "ImxpbmsiOjE3fSx7Im5hbWUiOiJhdWRpbyIsInR5cGUiOiJBVURJTyIsImxpbmsiOjE4LCJzaGFwZSI6N31dLCJvdXRwdXRzIjpb"
    "eyJuYW1lIjoiVklERU8iLCJ0eXBlIjoiVklERU8iLCJsaW5rcyI6WzE5XX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3Ig"
    "UyZSIjoiQ3JlYXRlVmlkZW8ifSwid2lkZ2V0c192YWx1ZXMiOlsyNCw4LCJzUkdCIl19LHsiaWQiOjE2LCJ0eXBlIjoiU2F2ZVZp"
    "ZGVvIiwicG9zIjpbMzIyMCwwXSwic2l6ZSI6WzQwMCwxMzRdLCJmbGFncyI6e30sIm9yZGVyIjoxNiwibW9kZSI6MCwiaW5wdXRz"
    "IjpbeyJuYW1lIjoidmlkZW8iLCJ0eXBlIjoiVklERU8iLCJsaW5rIjoxOX1dLCJvdXRwdXRzIjpbeyJuYW1lIjoidmlkZW8iLCJ0"
    "eXBlIjoiVklERU8iLCJsaW5rcyI6bnVsbH1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiU2F2ZVZpZGVvIn0s"
    "IndpZGdldHNfdmFsdWVzIjpbIlNpbmd1bGFyaXR5X1YyL3Npbmd1bGFyaXR5X3YyIiwiYXV0byIsImF1dG8iXX0seyJpZCI6MTcs"
    "InR5cGUiOiJMb2FkSW1hZ2UiLCJwb3MiOlswLDE4MjBdLCJzaXplIjpbNDAwLDYwXSwiZmxhZ3MiOnt9LCJvcmRlciI6NywibW9k"
    "ZSI6MCwiaW5wdXRzIjpbXSwib3V0cHV0cyI6W3sibmFtZSI6IklNQUdFIiwidHlwZSI6IklNQUdFIiwibGlua3MiOls1XX0seyJu"
    "YW1lIjoiTUFTSyIsInR5cGUiOiJNQVNLIiwibGlua3MiOm51bGx9XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6"
    "IkxvYWRJbWFnZSJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJwb3J0cmFpdF9waG90b19tYW5fYi5wbmciXX1dLCJsaW5rcyI6W1sxLDEs"
    "MCwyLDAsIk1PREVMIl0sWzIsMiwwLDMsMCwiTU9ERUwiXSxbMyw0LDAsNywwLCJDTElQIl0sWzQsNSwwLDcsMSwiVkFFIl0sWzUs"
    "MTcsMCw3LDIsIklNQUdFIl0sWzYsMywwLDksMCwiTU9ERUwiXSxbNyw3LDAsOSwxLCJDT05ESVRJT05JTkciXSxbOCw4LDAsMTIs"
    "MCwiTk9JU0UiXSxbOSw5LDAsMTIsMSwiR1VJREVSIl0sWzEwLDEwLDAsMTIsMiwiU0FNUExFUiJdLFsxMSwxMSwwLDEyLDMsIlNJ"
    "R01BUyJdLFsxMiw3LDEsMTIsNCwiTEFURU5UIl0sWzEzLDEyLDAsMTMsMCwiTEFURU5UIl0sWzE0LDUsMCwxMywxLCJWQUUiXSxb"
    "MTUsMTIsMCwxNCwwLCJMQVRFTlQiXSxbMTYsNiwwLDE0LDEsIlZBRSJdLFsxNywxMywwLDE1LDAsIklNQUdFIl0sWzE4LDE0LDAs"
    "MTUsMSwiQVVESU8iXSxbMTksMTUsMCwxNiwwLCJWSURFTyJdXSwiZ3JvdXBzIjpbXSwiY29uZmlnIjp7fSwiZXh0cmEiOnsiZHMi"
    "Onsic2NhbGUiOjAuNywib2Zmc2V0IjpbMCwwXX19LCJ2ZXJzaW9uIjowLjR9"
)
_WF_R2V_B64 = (
    "eyJpZCI6ImE3NjE3OGY4LTI2M2ItNDUzZC04YzdlLWJlZGNlNGVmZDllYyIsInJldmlzaW9uIjowLCJsYXN0X25vZGVfaWQiOjE3"
    "LCJsYXN0X2xpbmtfaWQiOjIwLCJub2RlcyI6W3siaWQiOjEsInR5cGUiOiJVTkVUTG9hZGVyIiwicG9zIjpbMCwwXSwic2l6ZSI6"
    "WzQwMCw4Ml0sImZsYWdzIjp7fSwib3JkZXIiOjAsIm1vZGUiOjAsImlucHV0cyI6W10sIm91dHB1dHMiOlt7Im5hbWUiOiJNT0RF"
    "TCIsInR5cGUiOiJNT0RFTCIsImxpbmtzIjpbMV19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlVORVRMb2Fk"
    "ZXIifSwid2lkZ2V0c192YWx1ZXMiOlsiTWluaW1heC1oM19TaW5ndWxhcml0eV9yZWYydmFfdjJfcHJ1bmVkX2ludDguc2FmZXRl"
    "bnNvcnMiLCJkZWZhdWx0Il0sInRpdGxlIjoiU2luZ3VsYXJpdHkgVjIgSU5UOCDjg6Ljg4fjg6sifSx7ImlkIjoyLCJ0eXBlIjoi"
    "TWluaU1heEgzU2lnbWFTaGlmdCIsInBvcyI6WzQ2MCwwXSwic2l6ZSI6WzQwMCwxMDhdLCJmbGFncyI6e30sIm9yZGVyIjo4LCJt"
    "b2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJtb2RlbCIsInR5cGUiOiJNT0RFTCIsImxpbmsiOjF9XSwib3V0cHV0cyI6W3sibmFt"
    "ZSI6Ik1PREVMIiwidHlwZSI6Ik1PREVMIiwibGlua3MiOlsyXX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoi"
    "TWluaU1heEgzU2lnbWFTaGlmdCJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WzEyLDNdLCJ0aXRsZSI6IuWLleeUu3NoaWZ0IDEy44O76Z+z"
    "5aOwc2hpZnQgMyJ9LHsiaWQiOjMsInR5cGUiOiJCbG9ja1NwYXJzZUF0dGVudGlvbiIsInBvcyI6WzkyMCwwXSwic2l6ZSI6WzQw"
    "MCwzNTBdLCJmbGFncyI6e30sIm9yZGVyIjoxMCwibW9kZSI6NCwiaW5wdXRzIjpbeyJuYW1lIjoibW9kZWwiLCJ0eXBlIjoiTU9E"
    "RUwiLCJsaW5rIjoyfV0sIm91dHB1dHMiOlt7Im5hbWUiOiJNT0RFTCIsInR5cGUiOiJNT0RFTCIsImxpbmtzIjpbNl19XSwicHJv"
    "cGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IkJsb2NrU3BhcnNlQXR0ZW50aW9uIn0sIndpZGdldHNfdmFsdWVzIjpbInZz"
    "YSIsMjAsMCwxLCIiLDAsMCwiZXhhY3Rfa3ZfYW5kX3Jvd3MiLHRydWVdLCJ3aWRnZXRzX3ZhbHVlc19uYW1lZCI6eyJzZWxlY3Rp"
    "b24ua2VlcF9wZXJjZW50IjoyMCwiZW5kX3BlcmNlbnQiOjEsImV4dHJhX3Rva2VucyI6MCwic3RhcnRfcGVyY2VudCI6MCwiZGVu"
    "c2VfYmxvY2tzIjoiIiwic2VsZWN0aW9uIjoidnNhIiwidmVyYm9zZSI6dHJ1ZSwic2lua19jb25kaXRpb25pbmciOiJleGFjdF9r"
    "dl9hbmRfcm93cyIsIm1pbl90b2tlbnMiOjB9LCJ0aXRsZSI6IueEoeWKue+8iOODkOOCpOODkeOCue+8iTog6YCa5bi4QXR0ZW50"
    "aW9u44Gn55Sf5oiQIn0seyJpZCI6NCwidHlwZSI6IkNMSVBMb2FkZXIiLCJwb3MiOlswLDI2MF0sInNpemUiOls0MDAsMTA4XSwi"
    "ZmxhZ3MiOnt9LCJvcmRlciI6MSwibW9kZSI6MCwiaW5wdXRzIjpbXSwib3V0cHV0cyI6W3sibmFtZSI6IkNMSVAiLCJ0eXBlIjoi"
    "Q0xJUCIsImxpbmtzIjpbM119XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IkNMSVBMb2FkZXIifSwid2lkZ2V0"
    "c192YWx1ZXMiOlsicXdlbjN2bF8zMmJfbWluaW1heF9oM19udmZwNF9hd3Euc2FmZXRlbnNvcnMiLCJtaW5pbWF4IiwiZGVmYXVs"
    "dCJdfSx7ImlkIjo1LCJ0eXBlIjoiVkFFTG9hZGVyIiwicG9zIjpbMCw1MjBdLCJzaXplIjpbNDAwLDYwXSwiZmxhZ3MiOnt9LCJv"
    "cmRlciI6MiwibW9kZSI6MCwiaW5wdXRzIjpbXSwib3V0cHV0cyI6W3sibmFtZSI6IlZBRSIsInR5cGUiOiJWQUUiLCJsaW5rcyI6"
    "WzQsMTRdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJWQUVMb2FkZXIifSwid2lkZ2V0c192YWx1ZXMiOlsi"
    "bWluaW1heF9oM192aWRlb192YWVfaW50OF9jb252cm90LnNhZmV0ZW5zb3JzIl19LHsiaWQiOjYsInR5cGUiOiJWQUVMb2FkZXIi"
    "LCJwb3MiOlswLDc4MF0sInNpemUiOls0MDAsNjBdLCJmbGFncyI6e30sIm9yZGVyIjozLCJtb2RlIjowLCJpbnB1dHMiOltdLCJv"
    "dXRwdXRzIjpbeyJuYW1lIjoiVkFFIiwidHlwZSI6IlZBRSIsImxpbmtzIjpbNSwxNl19XSwicHJvcGVydGllcyI6eyJOb2RlIG5h"
    "bWUgZm9yIFMmUiI6IlZBRUxvYWRlciJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJtaW5pbWF4X2gzX2F1ZGlvX3ZhZV9mcDMyLnNhZmV0"
    "ZW5zb3JzIl19LHsiaWQiOjcsInR5cGUiOiJNaW5pTWF4SDNSZWZlcmVuY2VUb1ZpZGVvIiwicG9zIjpbNDYwLDI2MF0sInNpemUi"
    "Ols0MDAsMzE2XSwiZmxhZ3MiOnt9LCJvcmRlciI6OSwibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoiY2xpcCIsInR5cGUiOiJD"
    "TElQIiwibGluayI6M30seyJuYW1lIjoidmFlIiwidHlwZSI6IlZBRSIsImxpbmsiOjR9LHsibmFtZSI6ImF1ZGlvX3ZhZSIsInR5"
    "cGUiOiJWQUUiLCJsaW5rIjo1fSx7Im5hbWUiOiJyZWZfaW1hZ2VzLnJlZl9pbWFnZV8wIiwidHlwZSI6IklNQUdFIiwibGluayI6"
    "MjB9XSwib3V0cHV0cyI6W3sibmFtZSI6InBvc2l0aXZlIiwidHlwZSI6IkNPTkRJVElPTklORyIsImxpbmtzIjpbN119LHsibmFt"
    "ZSI6IkxBVEVOVCIsInR5cGUiOiJMQVRFTlQiLCJsaW5rcyI6WzEyXX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZS"
    "IjoiTWluaU1heEgzUmVmZXJlbmNlVG9WaWRlbyJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJpbnRlZ3JhdGVkX211bHRpbW9kYWxfZGVz"
    "Y3JpcHRpb246IFtTaG90IDFdIDJELWFuaW1hdGVkLCBhbmltZSBzdHlsZSwgYSBtZWRpdW0gY2xvc2UtdXAgc2hvdCBmcmFtZXMg"
    "YSBjaGVlcmZ1bCB5b3VuZyB3b21hbiB3aXRoIGRhcmsgaGFpciB0aWVkIHVwIGluIGEgbmVhdCBidW4sIHNpZGUgYmFuZ3MgZnJh"
    "bWluZyBoZXIgZGVsaWNhdGUgZmFjZSwgYW5kIGRhcmsgcHVycGxlIGV5ZXMgd2VhcmluZyBzbWFsbCBwZWFybCBlYXJyaW5ncy4g"
    "U2hlIHdlYXJzIGEgbGlnaHQgZ3JheSB0dXJ0bGVuZWNrIGtuaXQgdG9wIHVuZGVyIGEgYmxhY2sgc3BhZ2hldHRpLXN0cmFwIG1p"
    "bmkgZHJlc3MuIFN0YW5kaW5nIGluIGEgY2xlYW4gbW9kZXJuIHJvb20gd2l0aCB3aGl0ZSB3YWxscyBhbmQgYSBkYXJrIGRvb3Jm"
    "cmFtZSwgc2hlIHRpbHRzIGhlciBoZWFkIHNsaWdodGx5IHdpdGggYSBicmlnaHQgc21pbGUsIGxpZ2h0bHkgcmVzdGluZyBoZXIg"
    "cmlnaHQgZmluZ2VydGlwcyBuZWFyIGhlciBjb2xsYXJib25lLiBUaGUgY2FtZXJhIHB1c2hlcyBpbiB3aXRoIHNtYWxsIGFtcGxp"
    "dHVkZSBhdCBzbG93IHNwZWVkIGFzIHRoZSB5b3VuZyB3b21hbiB3aXRoIGEgc3dlZXQsIGNsZWFyIHZvaWNlIChTMSkgZ2VudGx5"
    "IHNheXM6IDxkPltKYXBhbmVzZV0g44GT44KT44Gr44Gh44Gv77yB5LuK5pel44KC5LiA5pel6aCR5by144KN44GG44Gt44CCPC9k"
    "PlxuXG5vdmVyYWxsX3NvdW5kc2NhcGU6IFF1aWV0IGluZG9vciByb29tIHRvbmUsIGEgc29mdCBydXN0bGUgb2Yga25pdCBmYWJy"
    "aWMgYXMgc2hlIG1vdmVzIGhlciBzaG91bGRlciwgYW5kIHRoZSBjbGVhciBzcG9rZW4gdm9pY2UgaW4gYSBuYXR1cmFsIHJvb20g"
    "YWNvdXN0aWMuXG5cbm5vbl9kaWVnZXRpY19tdXNpYzogQSBnZW50bGUsIHVwbGlmdGluZyBhY291c3RpYyBndWl0YXIgcGF0dGVy"
    "biB3aXRoIHdhcm0gcGlhbm8gbm90ZXMgYXQgYSBtb2RlcmF0ZSB0ZW1wbywgZmFkaW5nIHNvZnRseSBhdCB0aGUgZW5kLiIsNTQ0"
    "LDgzMiwxMjQsIm1heCIsbnVsbCxudWxsLG51bGxdfSx7ImlkIjo4LCJ0eXBlIjoiUmFuZG9tTm9pc2UiLCJwb3MiOlswLDEwNDBd"
    "LCJzaXplIjpbNDAwLDYwXSwiZmxhZ3MiOnt9LCJvcmRlciI6NCwibW9kZSI6MCwiaW5wdXRzIjpbXSwib3V0cHV0cyI6W3sibmFt"
    "ZSI6Ik5PSVNFIiwidHlwZSI6Ik5PSVNFIiwibGlua3MiOls4XX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoi"
    "UmFuZG9tTm9pc2UifSwid2lkZ2V0c192YWx1ZXMiOls0Ml19LHsiaWQiOjksInR5cGUiOiJCYXNpY0d1aWRlciIsInBvcyI6WzEz"
    "ODAsMF0sInNpemUiOls0MDAsODJdLCJmbGFncyI6e30sIm9yZGVyIjoxMSwibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoibW9k"
    "ZWwiLCJ0eXBlIjoiTU9ERUwiLCJsaW5rIjo2fSx7Im5hbWUiOiJjb25kaXRpb25pbmciLCJ0eXBlIjoiQ09ORElUSU9OSU5HIiwi"
    "bGluayI6N31dLCJvdXRwdXRzIjpbeyJuYW1lIjoiR1VJREVSIiwidHlwZSI6IkdVSURFUiIsImxpbmtzIjpbOV19XSwicHJvcGVy"
    "dGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IkJhc2ljR3VpZGVyIn0sIndpZGdldHNfdmFsdWVzIjpbXX0seyJpZCI6MTAsInR5"
    "cGUiOiJLU2FtcGxlclNlbGVjdCIsInBvcyI6WzAsMTMwMF0sInNpemUiOls0MDAsNjBdLCJmbGFncyI6e30sIm9yZGVyIjo1LCJt"
    "b2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiU0FNUExFUiIsInR5cGUiOiJTQU1QTEVSIiwibGlua3MiOlsx"
    "MF19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IktTYW1wbGVyU2VsZWN0In0sIndpZGdldHNfdmFsdWVzIjpb"
    "ImV1bGVyIl19LHsiaWQiOjExLCJ0eXBlIjoiTWFudWFsU2lnbWFzIiwicG9zIjpbMCwxNTYwXSwic2l6ZSI6WzQwMCw2MF0sImZs"
    "YWdzIjp7fSwib3JkZXIiOjYsIm1vZGUiOjAsImlucHV0cyI6W10sIm91dHB1dHMiOlt7Im5hbWUiOiJTSUdNQVMiLCJ0eXBlIjoi"
    "U0lHTUFTIiwibGlua3MiOlsxMV19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6Ik1hbnVhbFNpZ21hcyJ9LCJ3"
    "aWRnZXRzX3ZhbHVlcyI6WyIwLjk5OTg5OTkwOTksIDAuOTk1NjMzMTg3OCwgMC45OTA4MjU2ODgxLCAwLjk4NTUwNzI0NjQsIDAu"
    "OTc5NTkxODM2NywgMC45NzI5NzI5NzMwLCAwLjk2NTUxNzI0MTQsIDAuOTU3MDU1MjE0NywgMC45NDczNjg0MjExLCAwLjkzNjE3"
    "MDIxMjgsIDAuOTIzMDc2OTIzMSwgMC45MDc1NjMwMjUyLCAwLjg4ODg4ODg4ODksIDAuODY1OTc5MzgxNCwgMC44MzcyMDkzMDIz"
    "LCAwLjgwMDAwMDAwMDAsIDAuNzUwMDAwMDAwMCwgMC42NzkyNDUyODMwLCAwLjU3MTQyODU3MTQsIDAuMzg3MDk2Nzc0MiwgMCJd"
    "LCJ0aXRsZSI6IlNpbmd1bGFyaXR5IFYyIDIw44K544OG44OD44OX77yI5q+U6LyD6ZaL5aeL6Kit5a6a77yJIn0seyJpZCI6MTIs"
    "InR5cGUiOiJTYW1wbGVyQ3VzdG9tQWR2YW5jZWQiLCJwb3MiOlsxODQwLDBdLCJzaXplIjpbNDAwLDE2MF0sImZsYWdzIjp7fSwi"
    "b3JkZXIiOjEyLCJtb2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJub2lzZSIsInR5cGUiOiJOT0lTRSIsImxpbmsiOjh9LHsibmFt"
    "ZSI6Imd1aWRlciIsInR5cGUiOiJHVUlERVIiLCJsaW5rIjo5fSx7Im5hbWUiOiJzYW1wbGVyIiwidHlwZSI6IlNBTVBMRVIiLCJs"
    "aW5rIjoxMH0seyJuYW1lIjoic2lnbWFzIiwidHlwZSI6IlNJR01BUyIsImxpbmsiOjExfSx7Im5hbWUiOiJsYXRlbnRfaW1hZ2Ui"
    "LCJ0eXBlIjoiTEFURU5UIiwibGluayI6MTJ9XSwib3V0cHV0cyI6W3sibmFtZSI6Im91dHB1dCIsInR5cGUiOiJMQVRFTlQiLCJs"
    "aW5rcyI6WzEzLDE1XX0seyJuYW1lIjoiZGVub2lzZWRfb3V0cHV0IiwidHlwZSI6IkxBVEVOVCIsImxpbmtzIjpudWxsfV0sInBy"
    "b3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJTYW1wbGVyQ3VzdG9tQWR2YW5jZWQifSwid2lkZ2V0c192YWx1ZXMiOltd"
    "fSx7ImlkIjoxMywidHlwZSI6IlZBRURlY29kZSIsInBvcyI6WzIzMDAsMF0sInNpemUiOls0MDAsODJdLCJmbGFncyI6e30sIm9y"
    "ZGVyIjoxMywibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoic2FtcGxlcyIsInR5cGUiOiJMQVRFTlQiLCJsaW5rIjoxM30seyJu"
    "YW1lIjoidmFlIiwidHlwZSI6IlZBRSIsImxpbmsiOjE0fV0sIm91dHB1dHMiOlt7Im5hbWUiOiJJTUFHRSIsInR5cGUiOiJJTUFH"
    "RSIsImxpbmtzIjpbMTddfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJWQUVEZWNvZGUifSwid2lkZ2V0c192"
    "YWx1ZXMiOltdfSx7ImlkIjoxNCwidHlwZSI6IlZBRURlY29kZUF1ZGlvIiwicG9zIjpbMjMwMCwyNjBdLCJzaXplIjpbNDAwLDgy"
    "XSwiZmxhZ3MiOnt9LCJvcmRlciI6MTQsIm1vZGUiOjAsImlucHV0cyI6W3sibmFtZSI6InNhbXBsZXMiLCJ0eXBlIjoiTEFURU5U"
    "IiwibGluayI6MTV9LHsibmFtZSI6InZhZSIsInR5cGUiOiJWQUUiLCJsaW5rIjoxNn1dLCJvdXRwdXRzIjpbeyJuYW1lIjoiQVVE"
    "SU8iLCJ0eXBlIjoiQVVESU8iLCJsaW5rcyI6WzE4XX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiVkFFRGVj"
    "b2RlQXVkaW8ifSwid2lkZ2V0c192YWx1ZXMiOltdfSx7ImlkIjoxNSwidHlwZSI6IkNyZWF0ZVZpZGVvIiwicG9zIjpbMjc2MCww"
    "XSwic2l6ZSI6WzQwMCwxNjBdLCJmbGFncyI6e30sIm9yZGVyIjoxNSwibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoiaW1hZ2Vz"
    "IiwidHlwZSI6IklNQUdFIiwibGluayI6MTd9LHsibmFtZSI6ImF1ZGlvIiwidHlwZSI6IkFVRElPIiwibGluayI6MTgsInNoYXBl"
    "Ijo3fV0sIm91dHB1dHMiOlt7Im5hbWUiOiJWSURFTyIsInR5cGUiOiJWSURFTyIsImxpbmtzIjpbMTldfV0sInByb3BlcnRpZXMi"
    "OnsiTm9kZSBuYW1lIGZvciBTJlIiOiJDcmVhdGVWaWRlbyJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WzI0LDgsInNSR0IiXX0seyJpZCI6"
    "MTYsInR5cGUiOiJTYXZlVmlkZW8iLCJwb3MiOlszMjIwLDBdLCJzaXplIjpbNDAwLDEzNF0sImZsYWdzIjp7fSwib3JkZXIiOjE2"
    "LCJtb2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJ2aWRlbyIsInR5cGUiOiJWSURFTyIsImxpbmsiOjE5fV0sIm91dHB1dHMiOlt7"
    "Im5hbWUiOiJ2aWRlbyIsInR5cGUiOiJWSURFTyIsImxpbmtzIjpudWxsfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBT"
    "JlIiOiJTYXZlVmlkZW8ifSwid2lkZ2V0c192YWx1ZXMiOlsiU2luZ3VsYXJpdHlfVjIvc2luZ3VsYXJpdHlfdjIiLCJhdXRvIiwi"
    "YXV0byJdfSx7ImlkIjoxNywidHlwZSI6IkxvYWRJbWFnZSIsInBvcyI6WzAsMTgyMF0sInNpemUiOls0MDAsNjBdLCJmbGFncyI6"
    "e30sIm9yZGVyIjo3LCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiSU1BR0UiLCJ0eXBlIjoiSU1BR0Ui"
    "LCJsaW5rcyI6WzIwXX0seyJuYW1lIjoiTUFTSyIsInR5cGUiOiJNQVNLIiwibGlua3MiOm51bGx9XSwicHJvcGVydGllcyI6eyJO"
    "b2RlIG5hbWUgZm9yIFMmUiI6IkxvYWRJbWFnZSJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJjaGFyc2hlZXRfYW5pbWVfaG9vZGllX2Jv"
    "eS5wbmciXX1dLCJsaW5rcyI6W1sxLDEsMCwyLDAsIk1PREVMIl0sWzIsMiwwLDMsMCwiTU9ERUwiXSxbMyw0LDAsNywwLCJDTElQ"
    "Il0sWzQsNSwwLDcsMSwiVkFFIl0sWzUsNiwwLDcsMiwiVkFFIl0sWzYsMywwLDksMCwiTU9ERUwiXSxbNyw3LDAsOSwxLCJDT05E"
    "SVRJT05JTkciXSxbOCw4LDAsMTIsMCwiTk9JU0UiXSxbOSw5LDAsMTIsMSwiR1VJREVSIl0sWzEwLDEwLDAsMTIsMiwiU0FNUExF"
    "UiJdLFsxMSwxMSwwLDEyLDMsIlNJR01BUyJdLFsxMiw3LDEsMTIsNCwiTEFURU5UIl0sWzEzLDEyLDAsMTMsMCwiTEFURU5UIl0s"
    "WzE0LDUsMCwxMywxLCJWQUUiXSxbMTUsMTIsMCwxNCwwLCJMQVRFTlQiXSxbMTYsNiwwLDE0LDEsIlZBRSJdLFsxNywxMywwLDE1"
    "LDAsIklNQUdFIl0sWzE4LDE0LDAsMTUsMSwiQVVESU8iXSxbMTksMTUsMCwxNiwwLCJWSURFTyJdLFsyMCwxNywwLDcsMywiSU1B"
    "R0UiXV0sImdyb3VwcyI6W10sImNvbmZpZyI6e30sImV4dHJhIjp7ImRzIjp7InNjYWxlIjowLjcsIm9mZnNldCI6WzAsMF19fSwi"
    "dmVyc2lvbiI6MC40fQ=="
)

_WF_R2V_APP_B64 = (
    "..."
    "c2lvbiI6MC40fQ=="
)

_FASTH3_WFS = {"singularity_v2_newvae_t2v.json": _WF_T2V_B64,
               "singularity_v2_newvae_i2v.json": _WF_I2V_B64,
               "singularity_v2_newvae_r2v.json": _WF_R2V_B64,
               "singularity_v2_newvae_r2v_app.json": _WF_R2V_APP_B64}

print("")
print("📥 Singularity V2のワークフローを配置します（UI形式・画面から開けます）...")
for _n, _b in _FASTH3_WFS.items():
    _p = _os.path.join(h3_workflow_dir, _n)
    with open(_p, "w", encoding="utf-8") as _f:
        _f.write(_b64.b64decode(_b).decode("utf-8"))
    _wf = _json.load(open(_p, encoding="utf-8"))
    print("  ✓ %s（ノード%d / 接続%d）" % (_n, len(_wf["nodes"]), len(_wf["links"])))


print('Singularity V2＋INT8映像VAE\n20ステップ・Euler・動画shift12／音声shift3。VSAはバイパスで無効。\nWorkflows > User workflows > Singularity_V2_NewVAE を選んでください。\nまずR2Vアプリ版の0.4MP・5秒で確認してください。Colab実機生成は未検証です。\n参照画像は接続されたものだけ有効です。動画は終了前に保存してください。')

# ========================================
# 🆕 サンプル画像の自動ダウンロード（inputフォルダへ格納）
# ========================================
print("\n📥 Downloading sample image to input folder...")
input_dir = "/content/ComfyUI/input"

# 参照画像・起点画像のサンプル（すべて架空のキャラクター／人物です）
model_download("https://raw.githubusercontent.com/zasuko/zasuko-fasth3-colab/main/samples/charsheet_anime_hoodie_boy.png", input_dir)   # アニメ調のキャラクターシート（R2V向け）
model_download("https://raw.githubusercontent.com/zasuko/zasuko-fasth3-colab/main/samples/charsheet_anime_vtuber_rion.png", input_dir)   # アニメ調・配色見本つき
model_download("https://raw.githubusercontent.com/zasuko/zasuko-fasth3-colab/main/samples/charsheet_anime_maid_girl.png", input_dir)   # アニメ調
model_download("https://raw.githubusercontent.com/zasuko/zasuko-fasth3-colab/main/samples/charsheet_photo_mascot_cat.png", input_dir)   # 実写調・着ぐるみ
model_download("https://raw.githubusercontent.com/zasuko/zasuko-fasth3-colab/main/samples/charsheet_photo_man_glasses.png", input_dir)   # 実写調
model_download("https://raw.githubusercontent.com/zasuko/zasuko-fasth3-colab/main/samples/charsheet_photo_woman_pink.png", input_dir)   # 実写調
model_download("https://raw.githubusercontent.com/zasuko/zasuko-fasth3-colab/main/samples/portrait_photo_man_a.png", input_dir)   # バストアップ（I2Vの起点向け）
model_download("https://raw.githubusercontent.com/zasuko/zasuko-fasth3-colab/main/samples/portrait_photo_man_b.png", input_dir)   # バストアップ（I2Vの起点向け）

print("✓ All sample images downloaded!\n")



# ========================================
# ComfyUI起動設定
# ========================================

# 🆕 出力ディレクトリの設定
output_dir_arg = ""
if use_google_drive and enable_gdrive_output:
    output_dir_arg = f"--output-directory {GDRIVE_OUTPUT}"
    print("=" * 70)
    print(f"✅ ComfyUI output will be saved directly to Google Drive")
    print(f"📁 Output path: {GDRIVE_OUTPUT}")
    print("=" * 70)

print("🚀 Pinggy Tunnel を準備しています...")
import subprocess
import threading
import time
import socket
import urllib.request
import html
import re
from IPython.display import HTML, display

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\nComfyUI finished loading... setting up Pinggy tunnel!\n")
    print("(Pinggyは、Colab上のComfyUIを外部から使えるようにする一時トンネルサービスです。下に出るURLはこのセッション限りの一時URLです)\n")

    # ★ トークンはセルの最初で読み込み済み。あればPro、無ければ無料（60分まで）でつなぐ。
    pinggy_target = f"{PINGGY_TOKEN}@pro.pinggy.io" if PINGGY_TOKEN else "a.pinggy.io"

    p = subprocess.Popen(["ssh", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=60", "-p", "443", "-R0:localhost:{}".format(port), pinggy_target], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stdout:
        l = line.decode()
        match = re.search(r"https://[^\s]+", l)
        if match and "dashboard.pinggy.io" not in match.group(0):
            comfyui_url = match.group(0).rstrip(".,;)]}")
            print("\n" + "="*70)
            print("🚀 Pinggy URL:", comfyui_url)
            print("="*70 + "\n")
            safe_url = html.escape(comfyui_url, quote=True)
            display(HTML(
                f'<a href="{safe_url}" target="_blank" '
                'style="display:inline-block;padding:12px 20px;background:#1976d2;color:white;'
                'font-weight:bold;text-decoration:none;border-radius:8px">ComfyUIを開く</a>'
            ))
            print("✅URLからリンクを開いたら Pinggy の赤色の🟥「Enter Site」のボタンをクリックして下さい。")
            if PINGGY_TOKEN:
                print("有料トークンで接続しているため、無料枠の60分制限ではなくPinggy Proの設定で動作します。")
            else:
                print("⚠️ 無料のPinggyで接続しています。60分でURLが切れるので、切れたらこのセルを再実行してください。")
            break

clear_output()
threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

# ★ 最重要修正箇所: --listen 0.0.0.0 と --enable-cors-header を追加して 403 Forbidden を回避します
# @markdown ---
# @markdown ## ⚡ Sage Attention（上級者向け、既定OFF推奨）
# @markdown このNotebookは通常Attentionで比較します。Sage Attentionは未検証のためOFFのまま使用してください。
use_sage_attention = False  # @param {type:"boolean"}

if use_sage_attention:
    raise RuntimeError("このSingularity比較版ではSage AttentionをOFFにしてください。")
_sage_flag = ""

if using_L4_GPU:
    command = f"python main.py {output_dir_arg} --listen 0.0.0.0 --enable-cors-header --cache-none{_sage_flag} --dont-print-server".strip()
    get_ipython().system(command)
else:
    command = f"python main.py {output_dir_arg} --listen 0.0.0.0 --enable-cors-header{_sage_flag} --dont-print-server".strip()
    get_ipython().system(command)



# 設定と出所（2026-10-26確認）

- 本体: `Minimax-h3_Singularity_ref2va_v2_pruned_int8.safetensors`（20,967,647,456バイト）。[固定版ダウンロード](https://huggingface.co/WarmBloodAban/Minimax-h3_Singularity/resolve/1259cf16cd0b655ba691a32fb46a409142762972/Minimax-h3_Singularity_ref2va_v2_pruned_int8.safetensors)
- モデルSHA256: `d7637c2c35cd30facd9e93778c491f6d82f91c4a441aecbdb4c384a62a5f3d9f`。配布メタデータとの照合用です。本Notebookは全ファイルのハッシュ照合を自動実行しません。
- INT8映像VAE（int8_convrot）: `minimax_h3_video_vae_int8_convrot.safetensors`。音声VAEとNVFP4-AWQ文章理解モデルは元Notebookの固定版を維持。
- ComfyUI v0.36.0 / `ee71d5c4993f29086b27fde1629a945ae48425bf`、PyTorch2.9.0/cu130、comfy-kitchen0.2.34、aimdo0.5.3を維持。
- 全4画面共通: Euler・20ステップ・動画shift12／音声shift3、BlockSparseAttentionノードはmode4（バイパス）で無効。
- ManualSigmas（計算の進み方）: `0.9998999099, 0.9956331878, 0.9908256881, 0.9855072464, 0.9795918367, 0.9729729730, 0.9655172414, 0.9570552147, 0.9473684211, 0.9361702128, 0.9230769231, 0.9075630252, 0.8888888889, 0.8659793814, 0.8372093023, 0.8000000000, 0.7500000000, 0.6792452830, 0.5714285714, 0.3870967742, 0`

## 利用入口と条件
- [Singularity作者の配布・説明](https://huggingface.co/WarmBloodAban/Minimax-h3_Singularity)
- [共通VAE・文章理解モデル](https://huggingface.co/Comfy-Org/MiniMax-H3)
- [元モデルのMiniMax H3利用条件](https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/LICENSE)
- [固定版ComfyUI](https://github.com/Comfy-Org/ComfyUI/tree/ee71d5c4993f29086b27fde1629a945ae48425bf)
- [Colab利用条件・制限](https://research.google.com/colaboratory/faq.html)
- [Pinggy](https://pinggy.io)

Singularityページのライセンスタグだけで、元モデルの地域・用途の条件が無くなるとは判断しないでください。配布元の条件を確認して利用します。ComfyUIはGPL-3.0です。

## 検査の限界
通常Attentionの小規模GPU計算を起動時に検査します。これだけでは量子化モデルの読み込み、長尺生成、画質、必要メモリを保証しません。
Colab実機未検証の派生Notebookです。エラー時はGPU名と最初のエラーを保存してください。VSAを有効に戻したりTurboを追加した結果は、この20回構成と別扱いです。
